In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import table
import io
import dataframe_image as dfi
import seaborn as sns

from pathlib import Path

# Define paths
DATA_PATH = Path("../../../../data/raw")

e:\parity-patterns-sri-lanka-main\parity-patterns-sri-lanka\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [3]:
# Load data
df = pd.read_excel(DATA_PATH / "2010_Birth_Final.xlsx")

In [4]:
# ============================================
# FOCUS ON BIRTH WEIGHT - ANALYZE MISSING PATTERNS
# ============================================
TARGET_VARIABLE = 'Birth_Weight(grams)'  # <-- Target variable to analyze
# ============================================

# ============================================
# COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING BIRTH WEIGHT PATTERNS
# WITH IUPAC COMPLIANT OUTPUT (Excel, Word, PNG)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
import os

warnings.filterwarnings('ignore')

# Create timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n{'='*80}")
print(f"📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING {TARGET_VARIABLE} PATTERNS")
print(f"(IUPAC Compliant Output)")
print(f"{'='*80}")

# Load the missing data
missing_data = df[df[TARGET_VARIABLE].isnull()].copy()
complete_data = df[df[TARGET_VARIABLE].notnull()].copy()

print(f"\n📌 DATASET OVERVIEW:")
print(f"   • Total records with missing {TARGET_VARIABLE}: {len(missing_data):,}")
print(f"   • Total records with complete {TARGET_VARIABLE}: {len(complete_data):,}")
print(f"   • Missing percentage: {(len(missing_data) / len(df)) * 100:.2f}%")

# ============================================
# 1. UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE")
print(f"{'='*80}")

# Create a comprehensive statistics dataframe
all_stats = []
significant_vars_list = []  # Store significant variables for Word document

# List all columns to analyze (exclude target variable)
columns_to_analyze = [col for col in missing_data.columns if col != TARGET_VARIABLE]

print(f"\nAnalyzing {len(columns_to_analyze)} variables...")

# Analyze each column
for col in columns_to_analyze:
    col_stats = {
        'Variable': col,
        'Data_Type': str(missing_data[col].dtype),
        'Missing_in_Group': missing_data[col].isnull().sum(),
        'Missing_in_Group_%': (missing_data[col].isnull().sum() / len(missing_data)) * 100,
        'Complete_in_Group': missing_data[col].notnull().sum(),
        'Complete_in_Group_%': (missing_data[col].notnull().sum() / len(missing_data)) * 100,
        'Unique_Values': missing_data[col].nunique(),
    }
    
    # Check if variable is numeric
    if missing_data[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(missing_data[col]):
        # NUMERIC VARIABLE STATISTICS
        valid_data = missing_data[col].dropna()
        complete_valid_data = complete_data[col].dropna() if col in complete_data.columns else pd.Series()
        
        if len(valid_data) > 0:
            if valid_data.nunique() > 1:
                try:
                    col_stats.update({
                        'Mean_Missing': valid_data.mean(),
                        'Median_Missing': valid_data.median(),
                        'Std_Dev_Missing': valid_data.std(),
                        'Min_Missing': valid_data.min(),
                        'Max_Missing': valid_data.max(),
                    })
                except:
                    col_stats.update({
                        'Mean_Missing': np.nan, 'Median_Missing': np.nan, 
                        'Std_Dev_Missing': np.nan, 'Min_Missing': np.nan, 'Max_Missing': np.nan
                    })
            else:
                constant_value = valid_data.iloc[0] if len(valid_data) > 0 else np.nan
                col_stats.update({
                    'Mean_Missing': constant_value, 'Median_Missing': constant_value, 
                    'Std_Dev_Missing': 0, 'Min_Missing': constant_value, 'Max_Missing': constant_value
                })
            
            # Compare with complete data
            if len(complete_valid_data) > 0:
                if len(valid_data) > 1 and len(complete_valid_data) > 1:
                    try:
                        t_stat, p_value = stats.ttest_ind(valid_data, complete_valid_data, equal_var=False)
                        col_stats['P_Value'] = p_value
                        col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                        col_stats['Mean_Difference'] = valid_data.mean() - complete_valid_data.mean()
                        
                        # Add complete group statistics
                        col_stats['Mean_Complete'] = complete_valid_data.mean()
                        col_stats['Median_Complete'] = complete_valid_data.median()
                        
                        # Store for detailed report
                        if p_value < 0.05:
                            significant_vars_list.append({
                                'Variable': col,
                                'Type': 'Numeric',
                                'Statistic': f"Mean={valid_data.mean():.2f}",
                                'Comparison': f"Complete Mean={complete_valid_data.mean():.2f}",
                                'Difference': col_stats['Mean_Difference'],
                                'Difference_%': (col_stats['Mean_Difference'] / abs(complete_valid_data.mean())) * 100 if complete_valid_data.mean() != 0 else np.nan,
                                'P_Value': p_value,
                                'Interpretation': f"Missing group has {abs(col_stats['Mean_Difference']):.2f} units {'higher' if col_stats['Mean_Difference'] > 0 else 'lower'} than complete group"
                            })
                    except:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'NO'
                        col_stats['Mean_Difference'] = np.nan
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    col_stats['Mean_Difference'] = np.nan
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
                col_stats['Mean_Difference'] = np.nan
        else:
            col_stats.update({
                'Mean_Missing': np.nan, 'Median_Missing': np.nan, 'Std_Dev_Missing': np.nan,
                'Min_Missing': np.nan, 'Max_Missing': np.nan, 'P_Value': np.nan,
                'Significant_Difference': 'N/A', 'Mean_Difference': np.nan
            })
            
    else:
        # CATEGORICAL VARIABLE STATISTICS (including race variables)
        valid_data = missing_data[col].dropna()
        
        if len(valid_data) > 0:
            value_counts = valid_data.value_counts()
            top_categories = value_counts.head(5)
            
            col_stats.update({
                'Most_Common': str(top_categories.index[0]) if len(top_categories) > 0 else np.nan,
                'Most_Common_%': (top_categories.iloc[0] / len(valid_data)) * 100 if len(top_categories) > 0 else 0,
                '2nd_Common': str(top_categories.index[1]) if len(top_categories) > 1 else np.nan,
                '2nd_Common_%': (top_categories.iloc[1] / len(valid_data)) * 100 if len(top_categories) > 1 else 0,
                '3rd_Common': str(top_categories.index[2]) if len(top_categories) > 2 else np.nan,
                '3rd_Common_%': (top_categories.iloc[2] / len(valid_data)) * 100 if len(top_categories) > 2 else 0,
            })
            
            # Chi-square test comparing with complete data
            if col in complete_data.columns:
                complete_valid = complete_data[col].dropna()
                if len(complete_valid) > 0:
                    try:
                        missing_cats = valid_data.value_counts()
                        complete_cats = complete_valid.value_counts()
                        
                        # Get all categories
                        all_cats = sorted(set(missing_cats.index) | set(complete_cats.index))
                        missing_counts = [missing_cats.get(cat, 0) for cat in all_cats]
                        complete_counts = [complete_cats.get(cat, 0) for cat in all_cats]
                        
                        # Only run chi-square if we have enough data
                        if len(all_cats) > 1 and sum(missing_counts) > 0 and sum(complete_counts) > 0:
                            contingency = np.array([missing_counts, complete_counts])
                            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
                            
                            col_stats['P_Value'] = p_value
                            col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                            
                            # Add complete group distribution
                            complete_top = complete_cats.index[0] if len(complete_cats) > 0 else 'N/A'
                            complete_top_pct = (complete_cats.iloc[0] / len(complete_valid)) * 100 if len(complete_cats) > 0 else 0
                            col_stats['Most_Common_Complete'] = complete_top
                            col_stats['Most_Common_Complete_%'] = complete_top_pct
                            
                            # Store for detailed report
                            if p_value < 0.05:
                                significant_vars_list.append({
                                    'Variable': col,
                                    'Type': 'Categorical',
                                    'Statistic': f"Most common='{col_stats['Most_Common']}' ({col_stats['Most_Common_%']:.1f}%)",
                                    'Comparison': f"Complete: Most common='{complete_top}' ({complete_top_pct:.1f}%)",
                                    'Difference': col_stats['Most_Common_%'] - complete_top_pct,
                                    'Difference_%': ((col_stats['Most_Common_%'] - complete_top_pct) / complete_top_pct * 100) if complete_top_pct != 0 else np.nan,
                                    'P_Value': p_value,
                                    'Interpretation': f"Missing group shows different distribution (p={p_value:.4f})"
                                })
                        else:
                            col_stats['P_Value'] = np.nan
                            col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    except Exception as e:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'ERROR'
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'N/A'
        else:
            col_stats.update({
                'Most_Common': np.nan, 'Most_Common_%': 0,
                '2nd_Common': np.nan, '2nd_Common_%': 0,
                '3rd_Common': np.nan, '3rd_Common_%': 0,
                'P_Value': np.nan, 'Significant_Difference': 'N/A'
            })
    
    all_stats.append(col_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_stats)

# Sort by significance
stats_df = stats_df.sort_values('P_Value', ascending=True)

print(f"\n✅ Analysis complete. Found {len(significant_vars_list)} variables with significant differences (p < 0.05)")

# Print significant variables in console
if significant_vars_list:
    print(f"\n📋 SIGNIFICANT VARIABLES (p < 0.05):")
    for var in significant_vars_list:
        if var['Type'] == 'Numeric':
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")
        else:
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

# ============================================
# 2. CREATE IUPAC COMPLIANT EXCEL REPORT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 2: CREATING IUPAC COMPLIANT EXCEL REPORT")
print(f"{'='*80}")

safe_filename_base = TARGET_VARIABLE.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
output_file = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.xlsx'

try:
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # Sheet 1: Variable-level Statistics
        stats_df.to_excel(writer, sheet_name='Variable_Statistics', index=False)
        
        # Sheet 2: Summary Statistics
        numeric_vars = [col for col in columns_to_analyze if missing_data[col].dtype in ['int64', 'float64']]
        categorical_vars = [col for col in columns_to_analyze if missing_data[col].dtype not in ['int64', 'float64']]
        
        summary_stats = pd.DataFrame({
            'Metric': [
                f'Total Missing {TARGET_VARIABLE} Records',
                f'Missing Percentage',
                'Total Variables Analyzed',
                'Numeric Variables',
                'Categorical Variables',
                'Variables with Significant Differences (p < 0.05)',
                'Analysis Date'
            ],
            'Value': [
                f"{len(missing_data):,}",
                f"{(len(missing_data) / len(df)) * 100:.2f}%",
                len(columns_to_analyze),
                len(numeric_vars),
                len(categorical_vars),
                len(significant_vars_list),
                datetime.now().strftime("%Y-%m-%d")
            ]
        })
        summary_stats.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 3: Variables with Significant Differences (Detailed)
        if significant_vars_list:
            sig_diff_df = pd.DataFrame(significant_vars_list)
            sig_diff_df.to_excel(writer, sheet_name='Significant_Differences_Detailed', index=False)
        
        # Format Excel sheets with IUPAC styling
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Auto-fit columns
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Style header row
            header_font = Font(name='Arial', size=10, bold=False, color='000000')
            header_fill = PatternFill(start_color='F0F0F0', end_color='F0F0F0', fill_type='solid')
            header_alignment = Alignment(horizontal='center', vertical='center')
            
            for cell in worksheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
            
            # Style data cells
            thin_border = Border(
                left=Side(style='thin', color='CCCCCC'),
                right=Side(style='thin', color='CCCCCC'),
                top=Side(style='thin', color='CCCCCC'),
                bottom=Side(style='thin', color='CCCCCC')
            )
            
            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name='Arial', size=9)
                    cell.border = thin_border
                    if isinstance(cell.value, (int, float)):
                        cell.alignment = Alignment(horizontal='right')
                    else:
                        cell.alignment = Alignment(horizontal='left')
    
    print(f"✅ Created IUPAC compliant Excel report: {output_file}")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

# ============================================
# 3. CREATE IUPAC COMPLIANT VISUALIZATIONS
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 3: CREATING IUPAC COMPLIANT VISUALIZATIONS")
print(f"{'='*80}")

# Set IUPAC style for matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8

# Create multi-panel figure
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
plot_count = 0

# 1. Bar plot for significant differences
if significant_vars_list:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    # Get top 15 significant variables
    top_vars = significant_vars_list[:15]
    var_names = [v['Variable'][:25] for v in top_vars]
    p_values = [-np.log10(v['P_Value']) for v in top_vars]
    
    bars = ax.barh(range(len(var_names)), p_values, color='#666666')
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('-log10(p-value)')
    ax.set_title('Top Variables by Significance Level')
    ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=0.5, label='p=0.05 threshold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend()

# 2. Missingness overview pie chart
plot_count += 1
ax = plt.subplot(2, 3, plot_count)

missing_pct = (len(missing_data) / len(df)) * 100
complete_pct = 100 - missing_pct
colors = ['#CCCCCC', '#666666']

wedges, texts, autotexts = ax.pie([missing_pct, complete_pct], 
                                    labels=[f'Missing\n({missing_pct:.1f}%)', 
                                            f'Complete\n({complete_pct:.1f}%)'],
                                    colors=colors,
                                    autopct='%1.1f%%',
                                    startangle=90,
                                    textprops={'fontsize': 9})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title(f'{TARGET_VARIABLE} Missingness Overview')

# 3. Effect sizes for numeric variables
numeric_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
if numeric_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in numeric_sig[:10]]
    differences = [v['Difference'] for v in numeric_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in differences]
    ax.barh(range(len(var_names)), differences, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Mean Difference')
    ax.set_title('Numeric Variables: Mean Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 4. Categorical variable distribution differences
categorical_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
if categorical_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in categorical_sig[:10]]
    diff_pct = [v['Difference_%'] if pd.notna(v['Difference_%']) else 0 for v in categorical_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in diff_pct]
    ax.barh(range(len(var_names)), diff_pct, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Difference in Most Common Category (%)')
    ax.set_title('Categorical Variables: Distribution Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 5. Summary text panel
plot_count += 1
ax = plt.subplot(2, 3, plot_count)
ax.axis('off')

sig_count = len(significant_vars_list)
total_vars = len(columns_to_analyze)
numeric_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Numeric'])
categorical_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Categorical'])

summary_text = f"IUPAC Statistical Summary\n\n"
summary_text += f"Target Variable: {TARGET_VARIABLE}\n"
summary_text += f"Missing rate: {missing_pct:.1f}%\n"
summary_text += f"Missing group: {len(missing_data):,} records\n"
summary_text += f"Complete group: {len(complete_data):,} records\n\n"
summary_text += f"Total variables analyzed: {total_vars}\n"
summary_text += f"Variables with significant differences: {sig_count}\n"
summary_text += f"  • Numeric: {numeric_sig_count}\n"
summary_text += f"  • Categorical: {categorical_sig_count}\n\n"

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN - Strong evidence of non-random missingness"
elif sig_count > 2:
    pattern = "MODERATE PATTERN - Some systematic patterns detected"
else:
    pattern = "RANDOM PATTERN - Missing appears relatively random"

summary_text += f"Missingness pattern: {pattern}\n\n"
summary_text += f"Generated: {datetime.now().strftime('%Y-%m-%d')}"

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', family='sans-serif',
        bbox=dict(boxstyle='round', facecolor='#F0F0F0', edgecolor='#CCCCCC'))

plt.suptitle(f'Missing {TARGET_VARIABLE} Analysis\nIUPAC Compliant Statistical Report', 
             fontsize=12, fontweight='normal', y=1.02)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save with IUPAC specifications
output_png = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.png'
try:
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print(f"✅ Created IUPAC compliant visualization: {output_png}")
except Exception as e:
    print(f"❌ Error saving PNG: {e}")

# ============================================
# 4. CREATE IUPAC COMPLIANT WORD DOCUMENT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 4: CREATING IUPAC COMPLIANT WORD DOCUMENT")
print(f"{'='*80}")

# Create Word document
doc = Document()

# Set document properties
doc.core_properties.author = "Data Analysis Report"
doc.core_properties.created = datetime.now()

# Add title
title = doc.add_heading(f'Missing {TARGET_VARIABLE} Analysis Report', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add metadata
doc.add_paragraph(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
doc.add_paragraph(f'Dataset: {len(df):,} rows × {len(df.columns)} columns')
doc.add_paragraph(f'Target Variable: {TARGET_VARIABLE}')
doc.add_paragraph()

# Executive Summary
doc.add_heading('1. Executive Summary', level=1)

summary_text = f"""
This report presents a comprehensive IUPAC-compliant statistical analysis of missing values in {TARGET_VARIABLE}.

Key findings include:
• Missing records: {len(missing_data):,} ({missing_pct:.1f}% of total data)
• Variables with significant differences (p < 0.05): {sig_count}
  - Numeric variables: {numeric_sig_count}
  - Categorical variables: {categorical_sig_count}
• Total variables analyzed: {total_vars}

Based on the statistical analysis, the missingness pattern is classified as:
"""
if sig_count > 5:
    summary_text += "\nSYSTEMATIC PATTERN - Missing data is strongly associated with other variables, suggesting non-random missingness."
elif sig_count > 2:
    summary_text += "\nMODERATE PATTERN - Some systematic patterns detected in the missing data."
else:
    summary_text += "\nRANDOM PATTERN - Missing data appears relatively random."

doc.add_paragraph(summary_text.strip())

# Statistical Summary
doc.add_heading('2. Statistical Summary', level=1)

# Create summary table
summary_table = doc.add_table(rows=len(summary_stats) + 1, cols=2)
summary_table.style = 'Light Grid Accent 1'
summary_table.autofit = False
summary_table.columns[0].width = Inches(4)
summary_table.columns[1].width = Inches(3)

# Header
header_cells = summary_table.rows[0].cells
header_cells[0].text = 'Metric'
header_cells[1].text = 'Value'
for cell in header_cells:
    cell.paragraphs[0].runs[0].font.bold = True
    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

# Data
for i, row in summary_stats.iterrows():
    cells = summary_table.rows[i + 1].cells
    cells[0].text = str(row['Metric'])
    cells[1].text = str(row['Value'])
    cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT

# ============================================
# DETAILED SIGNIFICANT DIFFERENCES SECTION
# ============================================

if significant_vars_list:
    doc.add_heading('3. Variables with Significant Differences (p < 0.05)', level=1)
    doc.add_paragraph(f'The following {len(significant_vars_list)} variables show statistically significant differences between missing and complete groups:')
    doc.add_paragraph()
    
    # Split into numeric and categorical
    numeric_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
    categorical_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
    
    # 3.1 Numeric Variables Section
    if numeric_vars_sig:
        doc.add_heading('3.1 Numeric Variables', level=2)
        
        # Create table for numeric variables
        num_table = doc.add_table(rows=len(numeric_vars_sig) + 1, cols=6)
        num_table.style = 'Light Grid Accent 1'
        num_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.2), Inches(1.2), Inches(1.2), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            num_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Mean', 'Complete Group Mean', 'Difference', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = num_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(numeric_vars_sig):
            row = num_table.rows[i + 1].cells
            row[0].text = var['Variable']
            
            # Extract mean from statistic string
            mean_val = var['Statistic'].split('=')[1].split()[0]
            row[1].text = mean_val
            
            # Extract complete mean
            comp_mean = var['Comparison'].split('=')[1]
            row[2].text = comp_mean
            
            row[3].text = f"{var['Difference']:+.2f}"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [1, 2, 3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.2 Categorical Variables Section (including race variables)
    if categorical_vars_sig:
        doc.add_heading('3.2 Categorical Variables', level=2)
        
        # Create table for categorical variables
        cat_table = doc.add_table(rows=len(categorical_vars_sig) + 1, cols=6)
        cat_table.style = 'Light Grid Accent 1'
        cat_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.5), Inches(1.5), Inches(1.0), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            cat_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Distribution', 'Complete Group Distribution', 'Difference (%)', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = cat_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(categorical_vars_sig):
            row = cat_table.rows[i + 1].cells
            row[0].text = var['Variable']
            row[1].text = var['Statistic']
            row[2].text = var['Comparison']
            row[3].text = f"{var['Difference']:+.1f}%" if pd.notna(var['Difference']) else "N/A"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.3 Summary of Key Findings
    doc.add_heading('3.3 Summary of Key Findings', level=2)
    
    findings_text = f"""
The analysis identified {len(significant_vars_list)} variables with statistically significant differences (p < 0.05):
• {len(numeric_vars_sig)} numeric variables show significant mean differences
• {len(categorical_vars_sig)} categorical variables show significant distribution differences

Top 10 Most Significant Variables:
"""
    
    # Add top 10 most significant findings
    sorted_vars = sorted(significant_vars_list, key=lambda x: x['P_Value'])[:10]
    for i, var in enumerate(sorted_vars, 1):
        if var['Type'] == 'Numeric':
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
        else:
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
    
    doc.add_paragraph(findings_text.strip())

# Add visualization
doc.add_heading('4. Statistical Visualizations', level=1)
if os.path.exists(output_png):
    doc.add_picture(output_png, width=Inches(6.5))
    doc.add_paragraph(f'Figure 1. IUPAC compliant statistical visualization of missing {TARGET_VARIABLE} patterns.')

# Recommendations
doc.add_heading('5. Recommendations', level=1)

if sig_count > 5:
    doc.add_paragraph('Based on the statistical analysis, the missing data exhibits SYSTEMATIC patterns:', style='List Bullet')
    doc.add_paragraph('Consider treating missing as a separate category in analysis', style='List Bullet')
    doc.add_paragraph('Use multiple imputation methods (MICE, EM algorithm)', style='List Bullet')
    doc.add_paragraph('Investigate the following key variables for common themes:', style='List Bullet')
    
    # Add list of significant variables
    for var in significant_vars_list[:10]:
        doc.add_paragraph(f'  - {var["Variable"]} (p={var["P_Value"]:.4f})', style='List Bullet')
    
    doc.add_paragraph('Document missingness patterns in methodology section', style='List Bullet')
    
elif sig_count > 2:
    doc.add_paragraph('Based on the statistical analysis, the missing data shows MODERATE systematic patterns:', style='List Bullet')
    doc.add_paragraph('Consider multiple imputation or sensitivity analysis', style='List Bullet')
    doc.add_paragraph('Document the patterns in your methodology', style='List Bullet')
    doc.add_paragraph('Validate findings with domain experts', style='List Bullet')
    
else:
    doc.add_paragraph('Based on the statistical analysis, the missing data appears RELATIVELY RANDOM:', style='List Bullet')
    doc.add_paragraph('Simple imputation methods may be acceptable (mean, median, mode)', style='List Bullet')
    doc.add_paragraph('Listwise deletion may be appropriate if missing rate is low', style='List Bullet')
    doc.add_paragraph('Still verify randomness with domain knowledge', style='List Bullet')

# Methodology
doc.add_heading('6. Methodology (IUPAC Compliant)', level=1)
doc.add_paragraph('Analysis performed according to IUPAC guidelines for data presentation:')
doc.add_paragraph('• Missing values reported as both count (n) and percentage (%)', style='List Bullet')
doc.add_paragraph('• Tables formatted with thin borders and alternating row shading', style='List Bullet')
doc.add_paragraph('• Sans-serif fonts (Arial) used for optimal readability', style='List Bullet')
doc.add_paragraph('• Statistical tests: t-test (numeric), χ² test (categorical)', style='List Bullet')
doc.add_paragraph('• Significance threshold: α = 0.05', style='List Bullet')
doc.add_paragraph('• Effect sizes reported where applicable', style='List Bullet')
doc.add_paragraph('• Figures saved at 300 DPI for publication quality', style='List Bullet')

# Add footer with page numbers
section = doc.sections[0]
footer = section.footer
footer_para = footer.paragraphs[0]
footer_para.text = f"{TARGET_VARIABLE} Missing Analysis | Generated: {datetime.now().strftime('%Y-%m-%d')} | Page "
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add page number field
run = footer_para.add_run()
fldChar1 = OxmlElement('w:fldChar')
fldChar1.set(qn('w:fldCharType'), 'begin')
run._r.append(fldChar1)

instrText = OxmlElement('w:instrText')
instrText.text = "PAGE"
run._r.append(instrText)

fldChar2 = OxmlElement('w:fldChar')
fldChar2.set(qn('w:fldCharType'), 'end')
run._r.append(fldChar2)

# Save document
output_docx = f'missing_{safe_filename_base}_iupac_report_{timestamp}.docx'
try:
    doc.save(output_docx)
    print(f"✅ Created IUPAC compliant Word document: {output_docx}")
except PermissionError as e:
    alt_filename = f'missing_{safe_filename_base}_iupac_report_{timestamp}_new.docx'
    doc.save(alt_filename)
    print(f"✅ Saved with alternative filename: {alt_filename}")
    output_docx = alt_filename

# ============================================
# 5. FINAL SUMMARY
# ============================================

print(f"\n{'='*80}")
print(f"📋 IUPAC COMPLIANT ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📊 FILES GENERATED (IUPAC Compliant):")
print(f"   • Excel Report: {output_file}")
print(f"   • Visualization: {output_png}")
print(f"   • Word Report: {output_docx}")

print(f"\n📈 STATISTICAL SUMMARY:")
print(f"   • Missing {TARGET_VARIABLE}: {len(missing_data):,} ({missing_pct:.1f}%)")
print(f"   • Variables with significant differences: {sig_count}")
print(f"     - Numeric: {numeric_sig_count}")
print(f"     - Categorical: {categorical_sig_count}")

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN"
elif sig_count > 2:
    pattern = "MODERATE PATTERN"
else:
    pattern = "RANDOM PATTERN"

print(f"   • Missingness pattern: {pattern}")

print(f"\n📋 TOP SIGNIFICANT VARIABLES:")
for var in significant_vars_list[:10]:
    print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

print(f"\n{'='*80}")
print(f"✅ All IUPAC compliant files generated successfully")
print(f"{'='*80}")


📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING Birth_Weight(grams) PATTERNS
(IUPAC Compliant Output)

📌 DATASET OVERVIEW:
   • Total records with missing Birth_Weight(grams): 45,303
   • Total records with complete Birth_Weight(grams): 318,578
   • Missing percentage: 12.45%

📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE

Analyzing 17 variables...

✅ Analysis complete. Found 14 variables with significant differences (p < 0.05)

📋 SIGNIFICANT VARIABLES (p < 0.05):
   • Registered_Month2.0: Mean=6.85, p=0.0000
   • Registered_Month: Most common='August' (12.6%), p=0.0000
   • Registered_District_2.0: Mean=37.78, p=0.0000
   • Registered_District: Most common='Kandy' (25.9%), p=0.0000
   • Birh_Year: Mean=2008.11, p=0.0000
   • Birth_Month: Most common='August' (10.6%), p=0.0000
   • Gender: Most common='Male' (50.6%), p=0.0041
   • Hospital or Not: Most common='Hospital' (92.8%), p=0.0000
   • Birth_Order: Most common='First' (56.4%), p=0.0000
   • Age of Mother

In [7]:
# ============================================
# FOCUS ON BIRTH WEIGHT - ANALYZE MISSING PATTERNS
# ============================================
TARGET_VARIABLE = 'Multiple_Birth_Status'  # <-- Target variable to analyze
# ============================================

# ============================================
# COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING BIRTH WEIGHT PATTERNS
# WITH IUPAC COMPLIANT OUTPUT (Excel, Word, PNG)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
import os

warnings.filterwarnings('ignore')

# Create timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n{'='*80}")
print(f"📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING {TARGET_VARIABLE} PATTERNS")
print(f"(IUPAC Compliant Output)")
print(f"{'='*80}")

# Load the missing data
missing_data = df[df[TARGET_VARIABLE].isnull()].copy()
complete_data = df[df[TARGET_VARIABLE].notnull()].copy()

print(f"\n📌 DATASET OVERVIEW:")
print(f"   • Total records with missing {TARGET_VARIABLE}: {len(missing_data):,}")
print(f"   • Total records with complete {TARGET_VARIABLE}: {len(complete_data):,}")
print(f"   • Missing percentage: {(len(missing_data) / len(df)) * 100:.2f}%")

# ============================================
# 1. UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE")
print(f"{'='*80}")

# Create a comprehensive statistics dataframe
all_stats = []
significant_vars_list = []  # Store significant variables for Word document

# List all columns to analyze (exclude target variable)
columns_to_analyze = [col for col in missing_data.columns if col != TARGET_VARIABLE]

print(f"\nAnalyzing {len(columns_to_analyze)} variables...")

# Analyze each column
for col in columns_to_analyze:
    col_stats = {
        'Variable': col,
        'Data_Type': str(missing_data[col].dtype),
        'Missing_in_Group': missing_data[col].isnull().sum(),
        'Missing_in_Group_%': (missing_data[col].isnull().sum() / len(missing_data)) * 100,
        'Complete_in_Group': missing_data[col].notnull().sum(),
        'Complete_in_Group_%': (missing_data[col].notnull().sum() / len(missing_data)) * 100,
        'Unique_Values': missing_data[col].nunique(),
    }
    
    # Check if variable is numeric
    if missing_data[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(missing_data[col]):
        # NUMERIC VARIABLE STATISTICS
        valid_data = missing_data[col].dropna()
        complete_valid_data = complete_data[col].dropna() if col in complete_data.columns else pd.Series()
        
        if len(valid_data) > 0:
            if valid_data.nunique() > 1:
                try:
                    col_stats.update({
                        'Mean_Missing': valid_data.mean(),
                        'Median_Missing': valid_data.median(),
                        'Std_Dev_Missing': valid_data.std(),
                        'Min_Missing': valid_data.min(),
                        'Max_Missing': valid_data.max(),
                    })
                except:
                    col_stats.update({
                        'Mean_Missing': np.nan, 'Median_Missing': np.nan, 
                        'Std_Dev_Missing': np.nan, 'Min_Missing': np.nan, 'Max_Missing': np.nan
                    })
            else:
                constant_value = valid_data.iloc[0] if len(valid_data) > 0 else np.nan
                col_stats.update({
                    'Mean_Missing': constant_value, 'Median_Missing': constant_value, 
                    'Std_Dev_Missing': 0, 'Min_Missing': constant_value, 'Max_Missing': constant_value
                })
            
            # Compare with complete data
            if len(complete_valid_data) > 0:
                if len(valid_data) > 1 and len(complete_valid_data) > 1:
                    try:
                        t_stat, p_value = stats.ttest_ind(valid_data, complete_valid_data, equal_var=False)
                        col_stats['P_Value'] = p_value
                        col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                        col_stats['Mean_Difference'] = valid_data.mean() - complete_valid_data.mean()
                        
                        # Add complete group statistics
                        col_stats['Mean_Complete'] = complete_valid_data.mean()
                        col_stats['Median_Complete'] = complete_valid_data.median()
                        
                        # Store for detailed report
                        if p_value < 0.05:
                            significant_vars_list.append({
                                'Variable': col,
                                'Type': 'Numeric',
                                'Statistic': f"Mean={valid_data.mean():.2f}",
                                'Comparison': f"Complete Mean={complete_valid_data.mean():.2f}",
                                'Difference': col_stats['Mean_Difference'],
                                'Difference_%': (col_stats['Mean_Difference'] / abs(complete_valid_data.mean())) * 100 if complete_valid_data.mean() != 0 else np.nan,
                                'P_Value': p_value,
                                'Interpretation': f"Missing group has {abs(col_stats['Mean_Difference']):.2f} units {'higher' if col_stats['Mean_Difference'] > 0 else 'lower'} than complete group"
                            })
                    except:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'NO'
                        col_stats['Mean_Difference'] = np.nan
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    col_stats['Mean_Difference'] = np.nan
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
                col_stats['Mean_Difference'] = np.nan
        else:
            col_stats.update({
                'Mean_Missing': np.nan, 'Median_Missing': np.nan, 'Std_Dev_Missing': np.nan,
                'Min_Missing': np.nan, 'Max_Missing': np.nan, 'P_Value': np.nan,
                'Significant_Difference': 'N/A', 'Mean_Difference': np.nan
            })
            
    else:
        # CATEGORICAL VARIABLE STATISTICS (including race variables)
        valid_data = missing_data[col].dropna()
        
        if len(valid_data) > 0:
            value_counts = valid_data.value_counts()
            top_categories = value_counts.head(5)
            
            col_stats.update({
                'Most_Common': str(top_categories.index[0]) if len(top_categories) > 0 else np.nan,
                'Most_Common_%': (top_categories.iloc[0] / len(valid_data)) * 100 if len(top_categories) > 0 else 0,
                '2nd_Common': str(top_categories.index[1]) if len(top_categories) > 1 else np.nan,
                '2nd_Common_%': (top_categories.iloc[1] / len(valid_data)) * 100 if len(top_categories) > 1 else 0,
                '3rd_Common': str(top_categories.index[2]) if len(top_categories) > 2 else np.nan,
                '3rd_Common_%': (top_categories.iloc[2] / len(valid_data)) * 100 if len(top_categories) > 2 else 0,
            })
            
            # Chi-square test comparing with complete data
            if col in complete_data.columns:
                complete_valid = complete_data[col].dropna()
                if len(complete_valid) > 0:
                    try:
                        missing_cats = valid_data.value_counts()
                        complete_cats = complete_valid.value_counts()
                        
                        # Get all categories
                        all_cats = sorted(set(missing_cats.index) | set(complete_cats.index))
                        missing_counts = [missing_cats.get(cat, 0) for cat in all_cats]
                        complete_counts = [complete_cats.get(cat, 0) for cat in all_cats]
                        
                        # Only run chi-square if we have enough data
                        if len(all_cats) > 1 and sum(missing_counts) > 0 and sum(complete_counts) > 0:
                            contingency = np.array([missing_counts, complete_counts])
                            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
                            
                            col_stats['P_Value'] = p_value
                            col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                            
                            # Add complete group distribution
                            complete_top = complete_cats.index[0] if len(complete_cats) > 0 else 'N/A'
                            complete_top_pct = (complete_cats.iloc[0] / len(complete_valid)) * 100 if len(complete_cats) > 0 else 0
                            col_stats['Most_Common_Complete'] = complete_top
                            col_stats['Most_Common_Complete_%'] = complete_top_pct
                            
                            # Store for detailed report
                            if p_value < 0.05:
                                significant_vars_list.append({
                                    'Variable': col,
                                    'Type': 'Categorical',
                                    'Statistic': f"Most common='{col_stats['Most_Common']}' ({col_stats['Most_Common_%']:.1f}%)",
                                    'Comparison': f"Complete: Most common='{complete_top}' ({complete_top_pct:.1f}%)",
                                    'Difference': col_stats['Most_Common_%'] - complete_top_pct,
                                    'Difference_%': ((col_stats['Most_Common_%'] - complete_top_pct) / complete_top_pct * 100) if complete_top_pct != 0 else np.nan,
                                    'P_Value': p_value,
                                    'Interpretation': f"Missing group shows different distribution (p={p_value:.4f})"
                                })
                        else:
                            col_stats['P_Value'] = np.nan
                            col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    except Exception as e:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'ERROR'
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'N/A'
        else:
            col_stats.update({
                'Most_Common': np.nan, 'Most_Common_%': 0,
                '2nd_Common': np.nan, '2nd_Common_%': 0,
                '3rd_Common': np.nan, '3rd_Common_%': 0,
                'P_Value': np.nan, 'Significant_Difference': 'N/A'
            })
    
    all_stats.append(col_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_stats)

# Sort by significance
stats_df = stats_df.sort_values('P_Value', ascending=True)

print(f"\n✅ Analysis complete. Found {len(significant_vars_list)} variables with significant differences (p < 0.05)")

# Print significant variables in console
if significant_vars_list:
    print(f"\n📋 SIGNIFICANT VARIABLES (p < 0.05):")
    for var in significant_vars_list:
        if var['Type'] == 'Numeric':
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")
        else:
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

# ============================================
# 2. CREATE IUPAC COMPLIANT EXCEL REPORT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 2: CREATING IUPAC COMPLIANT EXCEL REPORT")
print(f"{'='*80}")

safe_filename_base = TARGET_VARIABLE.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
output_file = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.xlsx'

try:
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # Sheet 1: Variable-level Statistics
        stats_df.to_excel(writer, sheet_name='Variable_Statistics', index=False)
        
        # Sheet 2: Summary Statistics
        numeric_vars = [col for col in columns_to_analyze if missing_data[col].dtype in ['int64', 'float64']]
        categorical_vars = [col for col in columns_to_analyze if missing_data[col].dtype not in ['int64', 'float64']]
        
        summary_stats = pd.DataFrame({
            'Metric': [
                f'Total Missing {TARGET_VARIABLE} Records',
                f'Missing Percentage',
                'Total Variables Analyzed',
                'Numeric Variables',
                'Categorical Variables',
                'Variables with Significant Differences (p < 0.05)',
                'Analysis Date'
            ],
            'Value': [
                f"{len(missing_data):,}",
                f"{(len(missing_data) / len(df)) * 100:.2f}%",
                len(columns_to_analyze),
                len(numeric_vars),
                len(categorical_vars),
                len(significant_vars_list),
                datetime.now().strftime("%Y-%m-%d")
            ]
        })
        summary_stats.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 3: Variables with Significant Differences (Detailed)
        if significant_vars_list:
            sig_diff_df = pd.DataFrame(significant_vars_list)
            sig_diff_df.to_excel(writer, sheet_name='Significant_Differences_Detailed', index=False)
        
        # Format Excel sheets with IUPAC styling
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Auto-fit columns
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Style header row
            header_font = Font(name='Arial', size=10, bold=False, color='000000')
            header_fill = PatternFill(start_color='F0F0F0', end_color='F0F0F0', fill_type='solid')
            header_alignment = Alignment(horizontal='center', vertical='center')
            
            for cell in worksheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
            
            # Style data cells
            thin_border = Border(
                left=Side(style='thin', color='CCCCCC'),
                right=Side(style='thin', color='CCCCCC'),
                top=Side(style='thin', color='CCCCCC'),
                bottom=Side(style='thin', color='CCCCCC')
            )
            
            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name='Arial', size=9)
                    cell.border = thin_border
                    if isinstance(cell.value, (int, float)):
                        cell.alignment = Alignment(horizontal='right')
                    else:
                        cell.alignment = Alignment(horizontal='left')
    
    print(f"✅ Created IUPAC compliant Excel report: {output_file}")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

# ============================================
# 3. CREATE IUPAC COMPLIANT VISUALIZATIONS
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 3: CREATING IUPAC COMPLIANT VISUALIZATIONS")
print(f"{'='*80}")

# Set IUPAC style for matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8

# Create multi-panel figure
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
plot_count = 0

# 1. Bar plot for significant differences
if significant_vars_list:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    # Get top 15 significant variables
    top_vars = significant_vars_list[:15]
    var_names = [v['Variable'][:25] for v in top_vars]
    p_values = [-np.log10(v['P_Value']) for v in top_vars]
    
    bars = ax.barh(range(len(var_names)), p_values, color='#666666')
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('-log10(p-value)')
    ax.set_title('Top Variables by Significance Level')
    ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=0.5, label='p=0.05 threshold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend()

# 2. Missingness overview pie chart
plot_count += 1
ax = plt.subplot(2, 3, plot_count)

missing_pct = (len(missing_data) / len(df)) * 100
complete_pct = 100 - missing_pct
colors = ['#CCCCCC', '#666666']

wedges, texts, autotexts = ax.pie([missing_pct, complete_pct], 
                                    labels=[f'Missing\n({missing_pct:.1f}%)', 
                                            f'Complete\n({complete_pct:.1f}%)'],
                                    colors=colors,
                                    autopct='%1.1f%%',
                                    startangle=90,
                                    textprops={'fontsize': 9})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title(f'{TARGET_VARIABLE} Missingness Overview')

# 3. Effect sizes for numeric variables
numeric_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
if numeric_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in numeric_sig[:10]]
    differences = [v['Difference'] for v in numeric_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in differences]
    ax.barh(range(len(var_names)), differences, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Mean Difference')
    ax.set_title('Numeric Variables: Mean Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 4. Categorical variable distribution differences
categorical_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
if categorical_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in categorical_sig[:10]]
    diff_pct = [v['Difference_%'] if pd.notna(v['Difference_%']) else 0 for v in categorical_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in diff_pct]
    ax.barh(range(len(var_names)), diff_pct, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Difference in Most Common Category (%)')
    ax.set_title('Categorical Variables: Distribution Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 5. Summary text panel
plot_count += 1
ax = plt.subplot(2, 3, plot_count)
ax.axis('off')

sig_count = len(significant_vars_list)
total_vars = len(columns_to_analyze)
numeric_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Numeric'])
categorical_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Categorical'])

summary_text = f"IUPAC Statistical Summary\n\n"
summary_text += f"Target Variable: {TARGET_VARIABLE}\n"
summary_text += f"Missing rate: {missing_pct:.1f}%\n"
summary_text += f"Missing group: {len(missing_data):,} records\n"
summary_text += f"Complete group: {len(complete_data):,} records\n\n"
summary_text += f"Total variables analyzed: {total_vars}\n"
summary_text += f"Variables with significant differences: {sig_count}\n"
summary_text += f"  • Numeric: {numeric_sig_count}\n"
summary_text += f"  • Categorical: {categorical_sig_count}\n\n"

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN - Strong evidence of non-random missingness"
elif sig_count > 2:
    pattern = "MODERATE PATTERN - Some systematic patterns detected"
else:
    pattern = "RANDOM PATTERN - Missing appears relatively random"

summary_text += f"Missingness pattern: {pattern}\n\n"
summary_text += f"Generated: {datetime.now().strftime('%Y-%m-%d')}"

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', family='sans-serif',
        bbox=dict(boxstyle='round', facecolor='#F0F0F0', edgecolor='#CCCCCC'))

plt.suptitle(f'Missing {TARGET_VARIABLE} Analysis\nIUPAC Compliant Statistical Report', 
             fontsize=12, fontweight='normal', y=1.02)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save with IUPAC specifications
output_png = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.png'
try:
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print(f"✅ Created IUPAC compliant visualization: {output_png}")
except Exception as e:
    print(f"❌ Error saving PNG: {e}")

# ============================================
# 4. CREATE IUPAC COMPLIANT WORD DOCUMENT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 4: CREATING IUPAC COMPLIANT WORD DOCUMENT")
print(f"{'='*80}")

# Create Word document
doc = Document()

# Set document properties
doc.core_properties.author = "Data Analysis Report"
doc.core_properties.created = datetime.now()

# Add title
title = doc.add_heading(f'Missing {TARGET_VARIABLE} Analysis Report', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add metadata
doc.add_paragraph(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
doc.add_paragraph(f'Dataset: {len(df):,} rows × {len(df.columns)} columns')
doc.add_paragraph(f'Target Variable: {TARGET_VARIABLE}')
doc.add_paragraph()

# Executive Summary
doc.add_heading('1. Executive Summary', level=1)

summary_text = f"""
This report presents a comprehensive IUPAC-compliant statistical analysis of missing values in {TARGET_VARIABLE}.

Key findings include:
• Missing records: {len(missing_data):,} ({missing_pct:.1f}% of total data)
• Variables with significant differences (p < 0.05): {sig_count}
  - Numeric variables: {numeric_sig_count}
  - Categorical variables: {categorical_sig_count}
• Total variables analyzed: {total_vars}

Based on the statistical analysis, the missingness pattern is classified as:
"""
if sig_count > 5:
    summary_text += "\nSYSTEMATIC PATTERN - Missing data is strongly associated with other variables, suggesting non-random missingness."
elif sig_count > 2:
    summary_text += "\nMODERATE PATTERN - Some systematic patterns detected in the missing data."
else:
    summary_text += "\nRANDOM PATTERN - Missing data appears relatively random."

doc.add_paragraph(summary_text.strip())

# Statistical Summary
doc.add_heading('2. Statistical Summary', level=1)

# Create summary table
summary_table = doc.add_table(rows=len(summary_stats) + 1, cols=2)
summary_table.style = 'Light Grid Accent 1'
summary_table.autofit = False
summary_table.columns[0].width = Inches(4)
summary_table.columns[1].width = Inches(3)

# Header
header_cells = summary_table.rows[0].cells
header_cells[0].text = 'Metric'
header_cells[1].text = 'Value'
for cell in header_cells:
    cell.paragraphs[0].runs[0].font.bold = True
    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

# Data
for i, row in summary_stats.iterrows():
    cells = summary_table.rows[i + 1].cells
    cells[0].text = str(row['Metric'])
    cells[1].text = str(row['Value'])
    cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT

# ============================================
# DETAILED SIGNIFICANT DIFFERENCES SECTION
# ============================================

if significant_vars_list:
    doc.add_heading('3. Variables with Significant Differences (p < 0.05)', level=1)
    doc.add_paragraph(f'The following {len(significant_vars_list)} variables show statistically significant differences between missing and complete groups:')
    doc.add_paragraph()
    
    # Split into numeric and categorical
    numeric_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
    categorical_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
    
    # 3.1 Numeric Variables Section
    if numeric_vars_sig:
        doc.add_heading('3.1 Numeric Variables', level=2)
        
        # Create table for numeric variables
        num_table = doc.add_table(rows=len(numeric_vars_sig) + 1, cols=6)
        num_table.style = 'Light Grid Accent 1'
        num_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.2), Inches(1.2), Inches(1.2), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            num_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Mean', 'Complete Group Mean', 'Difference', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = num_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(numeric_vars_sig):
            row = num_table.rows[i + 1].cells
            row[0].text = var['Variable']
            
            # Extract mean from statistic string
            mean_val = var['Statistic'].split('=')[1].split()[0]
            row[1].text = mean_val
            
            # Extract complete mean
            comp_mean = var['Comparison'].split('=')[1]
            row[2].text = comp_mean
            
            row[3].text = f"{var['Difference']:+.2f}"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [1, 2, 3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.2 Categorical Variables Section (including race variables)
    if categorical_vars_sig:
        doc.add_heading('3.2 Categorical Variables', level=2)
        
        # Create table for categorical variables
        cat_table = doc.add_table(rows=len(categorical_vars_sig) + 1, cols=6)
        cat_table.style = 'Light Grid Accent 1'
        cat_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.5), Inches(1.5), Inches(1.0), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            cat_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Distribution', 'Complete Group Distribution', 'Difference (%)', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = cat_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(categorical_vars_sig):
            row = cat_table.rows[i + 1].cells
            row[0].text = var['Variable']
            row[1].text = var['Statistic']
            row[2].text = var['Comparison']
            row[3].text = f"{var['Difference']:+.1f}%" if pd.notna(var['Difference']) else "N/A"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.3 Summary of Key Findings
    doc.add_heading('3.3 Summary of Key Findings', level=2)
    
    findings_text = f"""
The analysis identified {len(significant_vars_list)} variables with statistically significant differences (p < 0.05):
• {len(numeric_vars_sig)} numeric variables show significant mean differences
• {len(categorical_vars_sig)} categorical variables show significant distribution differences

Top 10 Most Significant Variables:
"""
    
    # Add top 10 most significant findings
    sorted_vars = sorted(significant_vars_list, key=lambda x: x['P_Value'])[:10]
    for i, var in enumerate(sorted_vars, 1):
        if var['Type'] == 'Numeric':
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
        else:
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
    
    doc.add_paragraph(findings_text.strip())

# Add visualization
doc.add_heading('4. Statistical Visualizations', level=1)
if os.path.exists(output_png):
    doc.add_picture(output_png, width=Inches(6.5))
    doc.add_paragraph(f'Figure 1. IUPAC compliant statistical visualization of missing {TARGET_VARIABLE} patterns.')

# Recommendations
doc.add_heading('5. Recommendations', level=1)

if sig_count > 5:
    doc.add_paragraph('Based on the statistical analysis, the missing data exhibits SYSTEMATIC patterns:', style='List Bullet')
    doc.add_paragraph('Consider treating missing as a separate category in analysis', style='List Bullet')
    doc.add_paragraph('Use multiple imputation methods (MICE, EM algorithm)', style='List Bullet')
    doc.add_paragraph('Investigate the following key variables for common themes:', style='List Bullet')
    
    # Add list of significant variables
    for var in significant_vars_list[:10]:
        doc.add_paragraph(f'  - {var["Variable"]} (p={var["P_Value"]:.4f})', style='List Bullet')
    
    doc.add_paragraph('Document missingness patterns in methodology section', style='List Bullet')
    
elif sig_count > 2:
    doc.add_paragraph('Based on the statistical analysis, the missing data shows MODERATE systematic patterns:', style='List Bullet')
    doc.add_paragraph('Consider multiple imputation or sensitivity analysis', style='List Bullet')
    doc.add_paragraph('Document the patterns in your methodology', style='List Bullet')
    doc.add_paragraph('Validate findings with domain experts', style='List Bullet')
    
else:
    doc.add_paragraph('Based on the statistical analysis, the missing data appears RELATIVELY RANDOM:', style='List Bullet')
    doc.add_paragraph('Simple imputation methods may be acceptable (mean, median, mode)', style='List Bullet')
    doc.add_paragraph('Listwise deletion may be appropriate if missing rate is low', style='List Bullet')
    doc.add_paragraph('Still verify randomness with domain knowledge', style='List Bullet')

# Methodology
doc.add_heading('6. Methodology (IUPAC Compliant)', level=1)
doc.add_paragraph('Analysis performed according to IUPAC guidelines for data presentation:')
doc.add_paragraph('• Missing values reported as both count (n) and percentage (%)', style='List Bullet')
doc.add_paragraph('• Tables formatted with thin borders and alternating row shading', style='List Bullet')
doc.add_paragraph('• Sans-serif fonts (Arial) used for optimal readability', style='List Bullet')
doc.add_paragraph('• Statistical tests: t-test (numeric), χ² test (categorical)', style='List Bullet')
doc.add_paragraph('• Significance threshold: α = 0.05', style='List Bullet')
doc.add_paragraph('• Effect sizes reported where applicable', style='List Bullet')
doc.add_paragraph('• Figures saved at 300 DPI for publication quality', style='List Bullet')

# Add footer with page numbers
section = doc.sections[0]
footer = section.footer
footer_para = footer.paragraphs[0]
footer_para.text = f"{TARGET_VARIABLE} Missing Analysis | Generated: {datetime.now().strftime('%Y-%m-%d')} | Page "
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add page number field
run = footer_para.add_run()
fldChar1 = OxmlElement('w:fldChar')
fldChar1.set(qn('w:fldCharType'), 'begin')
run._r.append(fldChar1)

instrText = OxmlElement('w:instrText')
instrText.text = "PAGE"
run._r.append(instrText)

fldChar2 = OxmlElement('w:fldChar')
fldChar2.set(qn('w:fldCharType'), 'end')
run._r.append(fldChar2)

# Save document
output_docx = f'missing_{safe_filename_base}_iupac_report_{timestamp}.docx'
try:
    doc.save(output_docx)
    print(f"✅ Created IUPAC compliant Word document: {output_docx}")
except PermissionError as e:
    alt_filename = f'missing_{safe_filename_base}_iupac_report_{timestamp}_new.docx'
    doc.save(alt_filename)
    print(f"✅ Saved with alternative filename: {alt_filename}")
    output_docx = alt_filename

# ============================================
# 5. FINAL SUMMARY
# ============================================

print(f"\n{'='*80}")
print(f"📋 IUPAC COMPLIANT ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📊 FILES GENERATED (IUPAC Compliant):")
print(f"   • Excel Report: {output_file}")
print(f"   • Visualization: {output_png}")
print(f"   • Word Report: {output_docx}")

print(f"\n📈 STATISTICAL SUMMARY:")
print(f"   • Missing {TARGET_VARIABLE}: {len(missing_data):,} ({missing_pct:.1f}%)")
print(f"   • Variables with significant differences: {sig_count}")
print(f"     - Numeric: {numeric_sig_count}")
print(f"     - Categorical: {categorical_sig_count}")

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN"
elif sig_count > 2:
    pattern = "MODERATE PATTERN"
else:
    pattern = "RANDOM PATTERN"

print(f"   • Missingness pattern: {pattern}")

print(f"\n📋 TOP SIGNIFICANT VARIABLES:")
for var in significant_vars_list[:10]:
    print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

print(f"\n{'='*80}")
print(f"✅ All IUPAC compliant files generated successfully")
print(f"{'='*80}")


📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING Multiple_Birth_Status PATTERNS
(IUPAC Compliant Output)

📌 DATASET OVERVIEW:
   • Total records with missing Multiple_Birth_Status: 363,824
   • Total records with complete Multiple_Birth_Status: 57
   • Missing percentage: 99.98%

📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE

Analyzing 17 variables...

✅ Analysis complete. Found 8 variables with significant differences (p < 0.05)

📋 SIGNIFICANT VARIABLES (p < 0.05):
   • Registered_Month: Most common='October' (8.9%), p=0.0000
   • Registered_District_2.0: Mean=41.60, p=0.0329
   • Registered_District: Most common='Colombo' (15.2%), p=0.0000
   • Birh_Year: Mean=2009.57, p=0.0000
   • Birth_Month: Most common='October' (9.3%), p=0.0038
   • Birth_Weight(grams): Mean=2887.63, p=0.0000
   • Birth_Order: Most common='First' (45.9%), p=0.0020
   • District_of_Mother: Most common='Colombo' (10.3%), p=0.0000

📌 PART 2: CREATING IUPAC COMPLIANT EXCEL REPORT
✅ Created I

In [8]:
# ============================================
# FOCUS ON BIRTH WEIGHT - ANALYZE MISSING PATTERNS
# ============================================
TARGET_VARIABLE = 'Race_of_Father'  # <-- Target variable to analyze
# ============================================

# ============================================
# COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING BIRTH WEIGHT PATTERNS
# WITH IUPAC COMPLIANT OUTPUT (Excel, Word, PNG)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
import os

warnings.filterwarnings('ignore')

# Create timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n{'='*80}")
print(f"📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING {TARGET_VARIABLE} PATTERNS")
print(f"(IUPAC Compliant Output)")
print(f"{'='*80}")

# Load the missing data
missing_data = df[df[TARGET_VARIABLE].isnull()].copy()
complete_data = df[df[TARGET_VARIABLE].notnull()].copy()

print(f"\n📌 DATASET OVERVIEW:")
print(f"   • Total records with missing {TARGET_VARIABLE}: {len(missing_data):,}")
print(f"   • Total records with complete {TARGET_VARIABLE}: {len(complete_data):,}")
print(f"   • Missing percentage: {(len(missing_data) / len(df)) * 100:.2f}%")

# ============================================
# 1. UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE")
print(f"{'='*80}")

# Create a comprehensive statistics dataframe
all_stats = []
significant_vars_list = []  # Store significant variables for Word document

# List all columns to analyze (exclude target variable)
columns_to_analyze = [col for col in missing_data.columns if col != TARGET_VARIABLE]

print(f"\nAnalyzing {len(columns_to_analyze)} variables...")

# Analyze each column
for col in columns_to_analyze:
    col_stats = {
        'Variable': col,
        'Data_Type': str(missing_data[col].dtype),
        'Missing_in_Group': missing_data[col].isnull().sum(),
        'Missing_in_Group_%': (missing_data[col].isnull().sum() / len(missing_data)) * 100,
        'Complete_in_Group': missing_data[col].notnull().sum(),
        'Complete_in_Group_%': (missing_data[col].notnull().sum() / len(missing_data)) * 100,
        'Unique_Values': missing_data[col].nunique(),
    }
    
    # Check if variable is numeric
    if missing_data[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(missing_data[col]):
        # NUMERIC VARIABLE STATISTICS
        valid_data = missing_data[col].dropna()
        complete_valid_data = complete_data[col].dropna() if col in complete_data.columns else pd.Series()
        
        if len(valid_data) > 0:
            if valid_data.nunique() > 1:
                try:
                    col_stats.update({
                        'Mean_Missing': valid_data.mean(),
                        'Median_Missing': valid_data.median(),
                        'Std_Dev_Missing': valid_data.std(),
                        'Min_Missing': valid_data.min(),
                        'Max_Missing': valid_data.max(),
                    })
                except:
                    col_stats.update({
                        'Mean_Missing': np.nan, 'Median_Missing': np.nan, 
                        'Std_Dev_Missing': np.nan, 'Min_Missing': np.nan, 'Max_Missing': np.nan
                    })
            else:
                constant_value = valid_data.iloc[0] if len(valid_data) > 0 else np.nan
                col_stats.update({
                    'Mean_Missing': constant_value, 'Median_Missing': constant_value, 
                    'Std_Dev_Missing': 0, 'Min_Missing': constant_value, 'Max_Missing': constant_value
                })
            
            # Compare with complete data
            if len(complete_valid_data) > 0:
                if len(valid_data) > 1 and len(complete_valid_data) > 1:
                    try:
                        t_stat, p_value = stats.ttest_ind(valid_data, complete_valid_data, equal_var=False)
                        col_stats['P_Value'] = p_value
                        col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                        col_stats['Mean_Difference'] = valid_data.mean() - complete_valid_data.mean()
                        
                        # Add complete group statistics
                        col_stats['Mean_Complete'] = complete_valid_data.mean()
                        col_stats['Median_Complete'] = complete_valid_data.median()
                        
                        # Store for detailed report
                        if p_value < 0.05:
                            significant_vars_list.append({
                                'Variable': col,
                                'Type': 'Numeric',
                                'Statistic': f"Mean={valid_data.mean():.2f}",
                                'Comparison': f"Complete Mean={complete_valid_data.mean():.2f}",
                                'Difference': col_stats['Mean_Difference'],
                                'Difference_%': (col_stats['Mean_Difference'] / abs(complete_valid_data.mean())) * 100 if complete_valid_data.mean() != 0 else np.nan,
                                'P_Value': p_value,
                                'Interpretation': f"Missing group has {abs(col_stats['Mean_Difference']):.2f} units {'higher' if col_stats['Mean_Difference'] > 0 else 'lower'} than complete group"
                            })
                    except:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'NO'
                        col_stats['Mean_Difference'] = np.nan
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    col_stats['Mean_Difference'] = np.nan
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
                col_stats['Mean_Difference'] = np.nan
        else:
            col_stats.update({
                'Mean_Missing': np.nan, 'Median_Missing': np.nan, 'Std_Dev_Missing': np.nan,
                'Min_Missing': np.nan, 'Max_Missing': np.nan, 'P_Value': np.nan,
                'Significant_Difference': 'N/A', 'Mean_Difference': np.nan
            })
            
    else:
        # CATEGORICAL VARIABLE STATISTICS (including race variables)
        valid_data = missing_data[col].dropna()
        
        if len(valid_data) > 0:
            value_counts = valid_data.value_counts()
            top_categories = value_counts.head(5)
            
            col_stats.update({
                'Most_Common': str(top_categories.index[0]) if len(top_categories) > 0 else np.nan,
                'Most_Common_%': (top_categories.iloc[0] / len(valid_data)) * 100 if len(top_categories) > 0 else 0,
                '2nd_Common': str(top_categories.index[1]) if len(top_categories) > 1 else np.nan,
                '2nd_Common_%': (top_categories.iloc[1] / len(valid_data)) * 100 if len(top_categories) > 1 else 0,
                '3rd_Common': str(top_categories.index[2]) if len(top_categories) > 2 else np.nan,
                '3rd_Common_%': (top_categories.iloc[2] / len(valid_data)) * 100 if len(top_categories) > 2 else 0,
            })
            
            # Chi-square test comparing with complete data
            if col in complete_data.columns:
                complete_valid = complete_data[col].dropna()
                if len(complete_valid) > 0:
                    try:
                        missing_cats = valid_data.value_counts()
                        complete_cats = complete_valid.value_counts()
                        
                        # Get all categories
                        all_cats = sorted(set(missing_cats.index) | set(complete_cats.index))
                        missing_counts = [missing_cats.get(cat, 0) for cat in all_cats]
                        complete_counts = [complete_cats.get(cat, 0) for cat in all_cats]
                        
                        # Only run chi-square if we have enough data
                        if len(all_cats) > 1 and sum(missing_counts) > 0 and sum(complete_counts) > 0:
                            contingency = np.array([missing_counts, complete_counts])
                            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
                            
                            col_stats['P_Value'] = p_value
                            col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                            
                            # Add complete group distribution
                            complete_top = complete_cats.index[0] if len(complete_cats) > 0 else 'N/A'
                            complete_top_pct = (complete_cats.iloc[0] / len(complete_valid)) * 100 if len(complete_cats) > 0 else 0
                            col_stats['Most_Common_Complete'] = complete_top
                            col_stats['Most_Common_Complete_%'] = complete_top_pct
                            
                            # Store for detailed report
                            if p_value < 0.05:
                                significant_vars_list.append({
                                    'Variable': col,
                                    'Type': 'Categorical',
                                    'Statistic': f"Most common='{col_stats['Most_Common']}' ({col_stats['Most_Common_%']:.1f}%)",
                                    'Comparison': f"Complete: Most common='{complete_top}' ({complete_top_pct:.1f}%)",
                                    'Difference': col_stats['Most_Common_%'] - complete_top_pct,
                                    'Difference_%': ((col_stats['Most_Common_%'] - complete_top_pct) / complete_top_pct * 100) if complete_top_pct != 0 else np.nan,
                                    'P_Value': p_value,
                                    'Interpretation': f"Missing group shows different distribution (p={p_value:.4f})"
                                })
                        else:
                            col_stats['P_Value'] = np.nan
                            col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    except Exception as e:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'ERROR'
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'N/A'
        else:
            col_stats.update({
                'Most_Common': np.nan, 'Most_Common_%': 0,
                '2nd_Common': np.nan, '2nd_Common_%': 0,
                '3rd_Common': np.nan, '3rd_Common_%': 0,
                'P_Value': np.nan, 'Significant_Difference': 'N/A'
            })
    
    all_stats.append(col_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_stats)

# Sort by significance
stats_df = stats_df.sort_values('P_Value', ascending=True)

print(f"\n✅ Analysis complete. Found {len(significant_vars_list)} variables with significant differences (p < 0.05)")

# Print significant variables in console
if significant_vars_list:
    print(f"\n📋 SIGNIFICANT VARIABLES (p < 0.05):")
    for var in significant_vars_list:
        if var['Type'] == 'Numeric':
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")
        else:
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

# ============================================
# 2. CREATE IUPAC COMPLIANT EXCEL REPORT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 2: CREATING IUPAC COMPLIANT EXCEL REPORT")
print(f"{'='*80}")

safe_filename_base = TARGET_VARIABLE.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
output_file = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.xlsx'

try:
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # Sheet 1: Variable-level Statistics
        stats_df.to_excel(writer, sheet_name='Variable_Statistics', index=False)
        
        # Sheet 2: Summary Statistics
        numeric_vars = [col for col in columns_to_analyze if missing_data[col].dtype in ['int64', 'float64']]
        categorical_vars = [col for col in columns_to_analyze if missing_data[col].dtype not in ['int64', 'float64']]
        
        summary_stats = pd.DataFrame({
            'Metric': [
                f'Total Missing {TARGET_VARIABLE} Records',
                f'Missing Percentage',
                'Total Variables Analyzed',
                'Numeric Variables',
                'Categorical Variables',
                'Variables with Significant Differences (p < 0.05)',
                'Analysis Date'
            ],
            'Value': [
                f"{len(missing_data):,}",
                f"{(len(missing_data) / len(df)) * 100:.2f}%",
                len(columns_to_analyze),
                len(numeric_vars),
                len(categorical_vars),
                len(significant_vars_list),
                datetime.now().strftime("%Y-%m-%d")
            ]
        })
        summary_stats.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 3: Variables with Significant Differences (Detailed)
        if significant_vars_list:
            sig_diff_df = pd.DataFrame(significant_vars_list)
            sig_diff_df.to_excel(writer, sheet_name='Significant_Differences_Detailed', index=False)
        
        # Format Excel sheets with IUPAC styling
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Auto-fit columns
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Style header row
            header_font = Font(name='Arial', size=10, bold=False, color='000000')
            header_fill = PatternFill(start_color='F0F0F0', end_color='F0F0F0', fill_type='solid')
            header_alignment = Alignment(horizontal='center', vertical='center')
            
            for cell in worksheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
            
            # Style data cells
            thin_border = Border(
                left=Side(style='thin', color='CCCCCC'),
                right=Side(style='thin', color='CCCCCC'),
                top=Side(style='thin', color='CCCCCC'),
                bottom=Side(style='thin', color='CCCCCC')
            )
            
            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name='Arial', size=9)
                    cell.border = thin_border
                    if isinstance(cell.value, (int, float)):
                        cell.alignment = Alignment(horizontal='right')
                    else:
                        cell.alignment = Alignment(horizontal='left')
    
    print(f"✅ Created IUPAC compliant Excel report: {output_file}")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

# ============================================
# 3. CREATE IUPAC COMPLIANT VISUALIZATIONS
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 3: CREATING IUPAC COMPLIANT VISUALIZATIONS")
print(f"{'='*80}")

# Set IUPAC style for matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8

# Create multi-panel figure
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
plot_count = 0

# 1. Bar plot for significant differences
if significant_vars_list:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    # Get top 15 significant variables
    top_vars = significant_vars_list[:15]
    var_names = [v['Variable'][:25] for v in top_vars]
    p_values = [-np.log10(v['P_Value']) for v in top_vars]
    
    bars = ax.barh(range(len(var_names)), p_values, color='#666666')
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('-log10(p-value)')
    ax.set_title('Top Variables by Significance Level')
    ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=0.5, label='p=0.05 threshold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend()

# 2. Missingness overview pie chart
plot_count += 1
ax = plt.subplot(2, 3, plot_count)

missing_pct = (len(missing_data) / len(df)) * 100
complete_pct = 100 - missing_pct
colors = ['#CCCCCC', '#666666']

wedges, texts, autotexts = ax.pie([missing_pct, complete_pct], 
                                    labels=[f'Missing\n({missing_pct:.1f}%)', 
                                            f'Complete\n({complete_pct:.1f}%)'],
                                    colors=colors,
                                    autopct='%1.1f%%',
                                    startangle=90,
                                    textprops={'fontsize': 9})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title(f'{TARGET_VARIABLE} Missingness Overview')

# 3. Effect sizes for numeric variables
numeric_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
if numeric_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in numeric_sig[:10]]
    differences = [v['Difference'] for v in numeric_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in differences]
    ax.barh(range(len(var_names)), differences, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Mean Difference')
    ax.set_title('Numeric Variables: Mean Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 4. Categorical variable distribution differences
categorical_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
if categorical_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in categorical_sig[:10]]
    diff_pct = [v['Difference_%'] if pd.notna(v['Difference_%']) else 0 for v in categorical_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in diff_pct]
    ax.barh(range(len(var_names)), diff_pct, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Difference in Most Common Category (%)')
    ax.set_title('Categorical Variables: Distribution Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 5. Summary text panel
plot_count += 1
ax = plt.subplot(2, 3, plot_count)
ax.axis('off')

sig_count = len(significant_vars_list)
total_vars = len(columns_to_analyze)
numeric_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Numeric'])
categorical_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Categorical'])

summary_text = f"IUPAC Statistical Summary\n\n"
summary_text += f"Target Variable: {TARGET_VARIABLE}\n"
summary_text += f"Missing rate: {missing_pct:.1f}%\n"
summary_text += f"Missing group: {len(missing_data):,} records\n"
summary_text += f"Complete group: {len(complete_data):,} records\n\n"
summary_text += f"Total variables analyzed: {total_vars}\n"
summary_text += f"Variables with significant differences: {sig_count}\n"
summary_text += f"  • Numeric: {numeric_sig_count}\n"
summary_text += f"  • Categorical: {categorical_sig_count}\n\n"

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN - Strong evidence of non-random missingness"
elif sig_count > 2:
    pattern = "MODERATE PATTERN - Some systematic patterns detected"
else:
    pattern = "RANDOM PATTERN - Missing appears relatively random"

summary_text += f"Missingness pattern: {pattern}\n\n"
summary_text += f"Generated: {datetime.now().strftime('%Y-%m-%d')}"

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', family='sans-serif',
        bbox=dict(boxstyle='round', facecolor='#F0F0F0', edgecolor='#CCCCCC'))

plt.suptitle(f'Missing {TARGET_VARIABLE} Analysis\nIUPAC Compliant Statistical Report', 
             fontsize=12, fontweight='normal', y=1.02)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save with IUPAC specifications
output_png = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.png'
try:
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print(f"✅ Created IUPAC compliant visualization: {output_png}")
except Exception as e:
    print(f"❌ Error saving PNG: {e}")

# ============================================
# 4. CREATE IUPAC COMPLIANT WORD DOCUMENT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 4: CREATING IUPAC COMPLIANT WORD DOCUMENT")
print(f"{'='*80}")

# Create Word document
doc = Document()

# Set document properties
doc.core_properties.author = "Data Analysis Report"
doc.core_properties.created = datetime.now()

# Add title
title = doc.add_heading(f'Missing {TARGET_VARIABLE} Analysis Report', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add metadata
doc.add_paragraph(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
doc.add_paragraph(f'Dataset: {len(df):,} rows × {len(df.columns)} columns')
doc.add_paragraph(f'Target Variable: {TARGET_VARIABLE}')
doc.add_paragraph()

# Executive Summary
doc.add_heading('1. Executive Summary', level=1)

summary_text = f"""
This report presents a comprehensive IUPAC-compliant statistical analysis of missing values in {TARGET_VARIABLE}.

Key findings include:
• Missing records: {len(missing_data):,} ({missing_pct:.1f}% of total data)
• Variables with significant differences (p < 0.05): {sig_count}
  - Numeric variables: {numeric_sig_count}
  - Categorical variables: {categorical_sig_count}
• Total variables analyzed: {total_vars}

Based on the statistical analysis, the missingness pattern is classified as:
"""
if sig_count > 5:
    summary_text += "\nSYSTEMATIC PATTERN - Missing data is strongly associated with other variables, suggesting non-random missingness."
elif sig_count > 2:
    summary_text += "\nMODERATE PATTERN - Some systematic patterns detected in the missing data."
else:
    summary_text += "\nRANDOM PATTERN - Missing data appears relatively random."

doc.add_paragraph(summary_text.strip())

# Statistical Summary
doc.add_heading('2. Statistical Summary', level=1)

# Create summary table
summary_table = doc.add_table(rows=len(summary_stats) + 1, cols=2)
summary_table.style = 'Light Grid Accent 1'
summary_table.autofit = False
summary_table.columns[0].width = Inches(4)
summary_table.columns[1].width = Inches(3)

# Header
header_cells = summary_table.rows[0].cells
header_cells[0].text = 'Metric'
header_cells[1].text = 'Value'
for cell in header_cells:
    cell.paragraphs[0].runs[0].font.bold = True
    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

# Data
for i, row in summary_stats.iterrows():
    cells = summary_table.rows[i + 1].cells
    cells[0].text = str(row['Metric'])
    cells[1].text = str(row['Value'])
    cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT

# ============================================
# DETAILED SIGNIFICANT DIFFERENCES SECTION
# ============================================

if significant_vars_list:
    doc.add_heading('3. Variables with Significant Differences (p < 0.05)', level=1)
    doc.add_paragraph(f'The following {len(significant_vars_list)} variables show statistically significant differences between missing and complete groups:')
    doc.add_paragraph()
    
    # Split into numeric and categorical
    numeric_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
    categorical_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
    
    # 3.1 Numeric Variables Section
    if numeric_vars_sig:
        doc.add_heading('3.1 Numeric Variables', level=2)
        
        # Create table for numeric variables
        num_table = doc.add_table(rows=len(numeric_vars_sig) + 1, cols=6)
        num_table.style = 'Light Grid Accent 1'
        num_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.2), Inches(1.2), Inches(1.2), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            num_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Mean', 'Complete Group Mean', 'Difference', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = num_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(numeric_vars_sig):
            row = num_table.rows[i + 1].cells
            row[0].text = var['Variable']
            
            # Extract mean from statistic string
            mean_val = var['Statistic'].split('=')[1].split()[0]
            row[1].text = mean_val
            
            # Extract complete mean
            comp_mean = var['Comparison'].split('=')[1]
            row[2].text = comp_mean
            
            row[3].text = f"{var['Difference']:+.2f}"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [1, 2, 3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.2 Categorical Variables Section (including race variables)
    if categorical_vars_sig:
        doc.add_heading('3.2 Categorical Variables', level=2)
        
        # Create table for categorical variables
        cat_table = doc.add_table(rows=len(categorical_vars_sig) + 1, cols=6)
        cat_table.style = 'Light Grid Accent 1'
        cat_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.5), Inches(1.5), Inches(1.0), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            cat_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Distribution', 'Complete Group Distribution', 'Difference (%)', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = cat_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(categorical_vars_sig):
            row = cat_table.rows[i + 1].cells
            row[0].text = var['Variable']
            row[1].text = var['Statistic']
            row[2].text = var['Comparison']
            row[3].text = f"{var['Difference']:+.1f}%" if pd.notna(var['Difference']) else "N/A"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.3 Summary of Key Findings
    doc.add_heading('3.3 Summary of Key Findings', level=2)
    
    findings_text = f"""
The analysis identified {len(significant_vars_list)} variables with statistically significant differences (p < 0.05):
• {len(numeric_vars_sig)} numeric variables show significant mean differences
• {len(categorical_vars_sig)} categorical variables show significant distribution differences

Top 10 Most Significant Variables:
"""
    
    # Add top 10 most significant findings
    sorted_vars = sorted(significant_vars_list, key=lambda x: x['P_Value'])[:10]
    for i, var in enumerate(sorted_vars, 1):
        if var['Type'] == 'Numeric':
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
        else:
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
    
    doc.add_paragraph(findings_text.strip())

# Add visualization
doc.add_heading('4. Statistical Visualizations', level=1)
if os.path.exists(output_png):
    doc.add_picture(output_png, width=Inches(6.5))
    doc.add_paragraph(f'Figure 1. IUPAC compliant statistical visualization of missing {TARGET_VARIABLE} patterns.')

# Recommendations
doc.add_heading('5. Recommendations', level=1)

if sig_count > 5:
    doc.add_paragraph('Based on the statistical analysis, the missing data exhibits SYSTEMATIC patterns:', style='List Bullet')
    doc.add_paragraph('Consider treating missing as a separate category in analysis', style='List Bullet')
    doc.add_paragraph('Use multiple imputation methods (MICE, EM algorithm)', style='List Bullet')
    doc.add_paragraph('Investigate the following key variables for common themes:', style='List Bullet')
    
    # Add list of significant variables
    for var in significant_vars_list[:10]:
        doc.add_paragraph(f'  - {var["Variable"]} (p={var["P_Value"]:.4f})', style='List Bullet')
    
    doc.add_paragraph('Document missingness patterns in methodology section', style='List Bullet')
    
elif sig_count > 2:
    doc.add_paragraph('Based on the statistical analysis, the missing data shows MODERATE systematic patterns:', style='List Bullet')
    doc.add_paragraph('Consider multiple imputation or sensitivity analysis', style='List Bullet')
    doc.add_paragraph('Document the patterns in your methodology', style='List Bullet')
    doc.add_paragraph('Validate findings with domain experts', style='List Bullet')
    
else:
    doc.add_paragraph('Based on the statistical analysis, the missing data appears RELATIVELY RANDOM:', style='List Bullet')
    doc.add_paragraph('Simple imputation methods may be acceptable (mean, median, mode)', style='List Bullet')
    doc.add_paragraph('Listwise deletion may be appropriate if missing rate is low', style='List Bullet')
    doc.add_paragraph('Still verify randomness with domain knowledge', style='List Bullet')

# Methodology
doc.add_heading('6. Methodology (IUPAC Compliant)', level=1)
doc.add_paragraph('Analysis performed according to IUPAC guidelines for data presentation:')
doc.add_paragraph('• Missing values reported as both count (n) and percentage (%)', style='List Bullet')
doc.add_paragraph('• Tables formatted with thin borders and alternating row shading', style='List Bullet')
doc.add_paragraph('• Sans-serif fonts (Arial) used for optimal readability', style='List Bullet')
doc.add_paragraph('• Statistical tests: t-test (numeric), χ² test (categorical)', style='List Bullet')
doc.add_paragraph('• Significance threshold: α = 0.05', style='List Bullet')
doc.add_paragraph('• Effect sizes reported where applicable', style='List Bullet')
doc.add_paragraph('• Figures saved at 300 DPI for publication quality', style='List Bullet')

# Add footer with page numbers
section = doc.sections[0]
footer = section.footer
footer_para = footer.paragraphs[0]
footer_para.text = f"{TARGET_VARIABLE} Missing Analysis | Generated: {datetime.now().strftime('%Y-%m-%d')} | Page "
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add page number field
run = footer_para.add_run()
fldChar1 = OxmlElement('w:fldChar')
fldChar1.set(qn('w:fldCharType'), 'begin')
run._r.append(fldChar1)

instrText = OxmlElement('w:instrText')
instrText.text = "PAGE"
run._r.append(instrText)

fldChar2 = OxmlElement('w:fldChar')
fldChar2.set(qn('w:fldCharType'), 'end')
run._r.append(fldChar2)

# Save document
output_docx = f'missing_{safe_filename_base}_iupac_report_{timestamp}.docx'
try:
    doc.save(output_docx)
    print(f"✅ Created IUPAC compliant Word document: {output_docx}")
except PermissionError as e:
    alt_filename = f'missing_{safe_filename_base}_iupac_report_{timestamp}_new.docx'
    doc.save(alt_filename)
    print(f"✅ Saved with alternative filename: {alt_filename}")
    output_docx = alt_filename

# ============================================
# 5. FINAL SUMMARY
# ============================================

print(f"\n{'='*80}")
print(f"📋 IUPAC COMPLIANT ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📊 FILES GENERATED (IUPAC Compliant):")
print(f"   • Excel Report: {output_file}")
print(f"   • Visualization: {output_png}")
print(f"   • Word Report: {output_docx}")

print(f"\n📈 STATISTICAL SUMMARY:")
print(f"   • Missing {TARGET_VARIABLE}: {len(missing_data):,} ({missing_pct:.1f}%)")
print(f"   • Variables with significant differences: {sig_count}")
print(f"     - Numeric: {numeric_sig_count}")
print(f"     - Categorical: {categorical_sig_count}")

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN"
elif sig_count > 2:
    pattern = "MODERATE PATTERN"
else:
    pattern = "RANDOM PATTERN"

print(f"   • Missingness pattern: {pattern}")

print(f"\n📋 TOP SIGNIFICANT VARIABLES:")
for var in significant_vars_list[:10]:
    print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

print(f"\n{'='*80}")
print(f"✅ All IUPAC compliant files generated successfully")
print(f"{'='*80}")


📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING Race_of_Father PATTERNS
(IUPAC Compliant Output)

📌 DATASET OVERVIEW:
   • Total records with missing Race_of_Father: 531
   • Total records with complete Race_of_Father: 363,350
   • Missing percentage: 0.15%

📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE

Analyzing 17 variables...

✅ Analysis complete. Found 10 variables with significant differences (p < 0.05)

📋 SIGNIFICANT VARIABLES (p < 0.05):
   • Registered_District_2.0: Mean=35.37, p=0.0000
   • Registered_District: Most common='Kalutara' (20.2%), p=0.0000
   • Birh_Year: Mean=2003.48, p=0.0000
   • Hospital or Not: Most common='Hospital' (87.8%), p=0.0000
   • Birth_Weight(grams): Mean=2691.05, p=0.0000
   • Birth_Order: Most common='First' (66.7%), p=0.0000
   • Age of Mother: Mean=27.14, p=0.0000
   • Marital_Status: Most common='Unmarried' (98.7%), p=0.0000
   • District_of_Mother: Most common='Kalutara' (15.8%), p=0.0000
   • Race_of_Mother: Most comm

In [9]:
# ============================================
# FOCUS ON BIRTH WEIGHT - ANALYZE MISSING PATTERNS
# ============================================
TARGET_VARIABLE = 'Multiple_Birth_Status'  # <-- Target variable to analyze
# ============================================

# ============================================
# COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING BIRTH WEIGHT PATTERNS
# WITH IUPAC COMPLIANT OUTPUT (Excel, Word, PNG)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
import os

warnings.filterwarnings('ignore')

# Create timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n{'='*80}")
print(f"📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING {TARGET_VARIABLE} PATTERNS")
print(f"(IUPAC Compliant Output)")
print(f"{'='*80}")

# Load the missing data
missing_data = df[df[TARGET_VARIABLE].isnull()].copy()
complete_data = df[df[TARGET_VARIABLE].notnull()].copy()

print(f"\n📌 DATASET OVERVIEW:")
print(f"   • Total records with missing {TARGET_VARIABLE}: {len(missing_data):,}")
print(f"   • Total records with complete {TARGET_VARIABLE}: {len(complete_data):,}")
print(f"   • Missing percentage: {(len(missing_data) / len(df)) * 100:.2f}%")

# ============================================
# 1. UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE")
print(f"{'='*80}")

# Create a comprehensive statistics dataframe
all_stats = []
significant_vars_list = []  # Store significant variables for Word document

# List all columns to analyze (exclude target variable)
columns_to_analyze = [col for col in missing_data.columns if col != TARGET_VARIABLE]

print(f"\nAnalyzing {len(columns_to_analyze)} variables...")

# Analyze each column
for col in columns_to_analyze:
    col_stats = {
        'Variable': col,
        'Data_Type': str(missing_data[col].dtype),
        'Missing_in_Group': missing_data[col].isnull().sum(),
        'Missing_in_Group_%': (missing_data[col].isnull().sum() / len(missing_data)) * 100,
        'Complete_in_Group': missing_data[col].notnull().sum(),
        'Complete_in_Group_%': (missing_data[col].notnull().sum() / len(missing_data)) * 100,
        'Unique_Values': missing_data[col].nunique(),
    }
    
    # Check if variable is numeric
    if missing_data[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(missing_data[col]):
        # NUMERIC VARIABLE STATISTICS
        valid_data = missing_data[col].dropna()
        complete_valid_data = complete_data[col].dropna() if col in complete_data.columns else pd.Series()
        
        if len(valid_data) > 0:
            if valid_data.nunique() > 1:
                try:
                    col_stats.update({
                        'Mean_Missing': valid_data.mean(),
                        'Median_Missing': valid_data.median(),
                        'Std_Dev_Missing': valid_data.std(),
                        'Min_Missing': valid_data.min(),
                        'Max_Missing': valid_data.max(),
                    })
                except:
                    col_stats.update({
                        'Mean_Missing': np.nan, 'Median_Missing': np.nan, 
                        'Std_Dev_Missing': np.nan, 'Min_Missing': np.nan, 'Max_Missing': np.nan
                    })
            else:
                constant_value = valid_data.iloc[0] if len(valid_data) > 0 else np.nan
                col_stats.update({
                    'Mean_Missing': constant_value, 'Median_Missing': constant_value, 
                    'Std_Dev_Missing': 0, 'Min_Missing': constant_value, 'Max_Missing': constant_value
                })
            
            # Compare with complete data
            if len(complete_valid_data) > 0:
                if len(valid_data) > 1 and len(complete_valid_data) > 1:
                    try:
                        t_stat, p_value = stats.ttest_ind(valid_data, complete_valid_data, equal_var=False)
                        col_stats['P_Value'] = p_value
                        col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                        col_stats['Mean_Difference'] = valid_data.mean() - complete_valid_data.mean()
                        
                        # Add complete group statistics
                        col_stats['Mean_Complete'] = complete_valid_data.mean()
                        col_stats['Median_Complete'] = complete_valid_data.median()
                        
                        # Store for detailed report
                        if p_value < 0.05:
                            significant_vars_list.append({
                                'Variable': col,
                                'Type': 'Numeric',
                                'Statistic': f"Mean={valid_data.mean():.2f}",
                                'Comparison': f"Complete Mean={complete_valid_data.mean():.2f}",
                                'Difference': col_stats['Mean_Difference'],
                                'Difference_%': (col_stats['Mean_Difference'] / abs(complete_valid_data.mean())) * 100 if complete_valid_data.mean() != 0 else np.nan,
                                'P_Value': p_value,
                                'Interpretation': f"Missing group has {abs(col_stats['Mean_Difference']):.2f} units {'higher' if col_stats['Mean_Difference'] > 0 else 'lower'} than complete group"
                            })
                    except:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'NO'
                        col_stats['Mean_Difference'] = np.nan
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    col_stats['Mean_Difference'] = np.nan
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
                col_stats['Mean_Difference'] = np.nan
        else:
            col_stats.update({
                'Mean_Missing': np.nan, 'Median_Missing': np.nan, 'Std_Dev_Missing': np.nan,
                'Min_Missing': np.nan, 'Max_Missing': np.nan, 'P_Value': np.nan,
                'Significant_Difference': 'N/A', 'Mean_Difference': np.nan
            })
            
    else:
        # CATEGORICAL VARIABLE STATISTICS (including race variables)
        valid_data = missing_data[col].dropna()
        
        if len(valid_data) > 0:
            value_counts = valid_data.value_counts()
            top_categories = value_counts.head(5)
            
            col_stats.update({
                'Most_Common': str(top_categories.index[0]) if len(top_categories) > 0 else np.nan,
                'Most_Common_%': (top_categories.iloc[0] / len(valid_data)) * 100 if len(top_categories) > 0 else 0,
                '2nd_Common': str(top_categories.index[1]) if len(top_categories) > 1 else np.nan,
                '2nd_Common_%': (top_categories.iloc[1] / len(valid_data)) * 100 if len(top_categories) > 1 else 0,
                '3rd_Common': str(top_categories.index[2]) if len(top_categories) > 2 else np.nan,
                '3rd_Common_%': (top_categories.iloc[2] / len(valid_data)) * 100 if len(top_categories) > 2 else 0,
            })
            
            # Chi-square test comparing with complete data
            if col in complete_data.columns:
                complete_valid = complete_data[col].dropna()
                if len(complete_valid) > 0:
                    try:
                        missing_cats = valid_data.value_counts()
                        complete_cats = complete_valid.value_counts()
                        
                        # Get all categories
                        all_cats = sorted(set(missing_cats.index) | set(complete_cats.index))
                        missing_counts = [missing_cats.get(cat, 0) for cat in all_cats]
                        complete_counts = [complete_cats.get(cat, 0) for cat in all_cats]
                        
                        # Only run chi-square if we have enough data and at least 2 categories
                        if len(all_cats) > 1 and sum(missing_counts) > 0 and sum(complete_counts) > 0:
                            # Check if any expected frequencies are < 5 (if so, use Fisher's exact or report warning)
                            contingency = np.array([missing_counts, complete_counts])
                            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
                            
                            col_stats['P_Value'] = p_value
                            col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                            
                            # Add complete group distribution
                            complete_top = complete_cats.index[0] if len(complete_cats) > 0 else 'N/A'
                            complete_top_pct = (complete_cats.iloc[0] / len(complete_valid)) * 100 if len(complete_cats) > 0 else 0
                            col_stats['Most_Common_Complete'] = complete_top
                            col_stats['Most_Common_Complete_%'] = complete_top_pct
                            
                            # Calculate additional metrics for categorical variables
                            missing_pct_by_cat = {}
                            for cat in all_cats:
                                missing_count = missing_cats.get(cat, 0)
                                complete_count = complete_cats.get(cat, 0)
                                total = missing_count + complete_count
                                if total > 0:
                                    missing_pct_by_cat[cat] = (missing_count / total) * 100
                            
                            col_stats['Missing_Rate_by_Category'] = missing_pct_by_cat
                            
                            # Store for detailed report
                            if p_value < 0.05:
                                # Calculate the difference in proportions for the most common category
                                diff_pct = col_stats['Most_Common_%'] - complete_top_pct
                                
                                significant_vars_list.append({
                                    'Variable': col,
                                    'Type': 'Categorical',
                                    'Statistic': f"Most common='{col_stats['Most_Common']}' ({col_stats['Most_Common_%']:.1f}%)",
                                    'Comparison': f"Complete: Most common='{complete_top}' ({complete_top_pct:.1f}%)",
                                    'Difference': diff_pct,
                                    'Difference_%': (diff_pct / complete_top_pct * 100) if complete_top_pct != 0 else np.nan,
                                    'P_Value': p_value,
                                    'Interpretation': f"Missing group shows different distribution (p={p_value:.4f})",
                                    'Chi2_Statistic': chi2,
                                    'Degrees_of_Freedom': dof
                                })
                        else:
                            col_stats['P_Value'] = np.nan
                            col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    except Exception as e:
                        print(f"   Warning: Error analyzing {col}: {e}")
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'ERROR'
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'N/A'
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
        else:
            col_stats.update({
                'Most_Common': np.nan, 'Most_Common_%': 0,
                '2nd_Common': np.nan, '2nd_Common_%': 0,
                '3rd_Common': np.nan, '3rd_Common_%': 0,
                'P_Value': np.nan, 'Significant_Difference': 'N/A'
            })
    
    all_stats.append(col_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_stats)

# Sort by significance
stats_df = stats_df.sort_values('P_Value', ascending=True)

print(f"\n✅ Analysis complete. Found {len(significant_vars_list)} variables with significant differences (p < 0.05)")

# Print significant variables in console
if significant_vars_list:
    print(f"\n📋 SIGNIFICANT VARIABLES (p < 0.05):")
    for var in significant_vars_list:
        if var['Type'] == 'Numeric':
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")
        else:
            print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

# ============================================
# ADDITIONAL ANALYSIS: DETAILED CATEGORICAL ANALYSIS FOR RACE VARIABLES
# ============================================

print(f"\n{'='*80}")
print(f"📌 ADDITIONAL ANALYSIS: DETAILED CATEGORICAL BREAKDOWN")
print(f"{'='*80}")

# Specifically analyze race variables with more detail
race_vars = [col for col in columns_to_analyze if 'Race' in col or 'race' in col]
for race_var in race_vars:
    if race_var in missing_data.columns:
        print(f"\n📊 Detailed Analysis for {race_var}:")
        
        missing_valid = missing_data[race_var].dropna()
        complete_valid = complete_data[race_var].dropna()
        
        if len(missing_valid) > 0 and len(complete_valid) > 0:
            # Get distributions
            missing_dist = missing_valid.value_counts(normalize=True) * 100
            complete_dist = complete_valid.value_counts(normalize=True) * 100
            
            print(f"\n   Distribution in MISSING group:")
            for category, pct in missing_dist.head(5).items():
                print(f"      • {category}: {pct:.1f}%")
            
            print(f"\n   Distribution in COMPLETE group:")
            for category, pct in complete_dist.head(5).items():
                print(f"      • {category}: {pct:.1f}%")
            
            # Calculate missing rate for each category
            all_categories = sorted(set(missing_dist.index) | set(complete_dist.index))
            print(f"\n   Missing rate by category:")
            for category in all_categories:
                missing_count = (missing_data[race_var] == category).sum()
                complete_count = (complete_data[race_var] == category).sum()
                total = missing_count + complete_count
                if total > 0:
                    missing_rate = (missing_count / total) * 100
                    print(f"      • {category}: {missing_count:,}/{total:,} ({missing_rate:.1f}%)")

# ============================================
# 2. CREATE IUPAC COMPLIANT EXCEL REPORT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 2: CREATING IUPAC COMPLIANT EXCEL REPORT")
print(f"{'='*80}")

safe_filename_base = TARGET_VARIABLE.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
output_file = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.xlsx'

try:
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # Sheet 1: Variable-level Statistics
        stats_df.to_excel(writer, sheet_name='Variable_Statistics', index=False)
        
        # Sheet 2: Summary Statistics
        numeric_vars = [col for col in columns_to_analyze if missing_data[col].dtype in ['int64', 'float64']]
        categorical_vars = [col for col in columns_to_analyze if missing_data[col].dtype not in ['int64', 'float64']]
        
        summary_stats = pd.DataFrame({
            'Metric': [
                f'Total Missing {TARGET_VARIABLE} Records',
                f'Missing Percentage',
                'Total Variables Analyzed',
                'Numeric Variables',
                'Categorical Variables',
                'Variables with Significant Differences (p < 0.05)',
                'Analysis Date'
            ],
            'Value': [
                f"{len(missing_data):,}",
                f"{(len(missing_data) / len(df)) * 100:.2f}%",
                len(columns_to_analyze),
                len(numeric_vars),
                len(categorical_vars),
                len(significant_vars_list),
                datetime.now().strftime("%Y-%m-%d")
            ]
        })
        summary_stats.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 3: Variables with Significant Differences (Detailed)
        if significant_vars_list:
            sig_diff_df = pd.DataFrame(significant_vars_list)
            sig_diff_df.to_excel(writer, sheet_name='Significant_Differences_Detailed', index=False)
        
        # Format Excel sheets with IUPAC styling
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Auto-fit columns
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Style header row
            header_font = Font(name='Arial', size=10, bold=False, color='000000')
            header_fill = PatternFill(start_color='F0F0F0', end_color='F0F0F0', fill_type='solid')
            header_alignment = Alignment(horizontal='center', vertical='center')
            
            for cell in worksheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
            
            # Style data cells
            thin_border = Border(
                left=Side(style='thin', color='CCCCCC'),
                right=Side(style='thin', color='CCCCCC'),
                top=Side(style='thin', color='CCCCCC'),
                bottom=Side(style='thin', color='CCCCCC')
            )
            
            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name='Arial', size=9)
                    cell.border = thin_border
                    if isinstance(cell.value, (int, float)):
                        cell.alignment = Alignment(horizontal='right')
                    else:
                        cell.alignment = Alignment(horizontal='left')
    
    print(f"✅ Created IUPAC compliant Excel report: {output_file}")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

# ============================================
# 3. CREATE IUPAC COMPLIANT VISUALIZATIONS
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 3: CREATING IUPAC COMPLIANT VISUALIZATIONS")
print(f"{'='*80}")

# Set IUPAC style for matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8

# Determine number of plots needed
num_plots = min(6, 2 + len(significant_vars_list) // 5)
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
plot_count = 0

# 1. Bar plot for significant differences
if significant_vars_list:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    # Get top 15 significant variables
    top_vars = significant_vars_list[:15]
    var_names = [v['Variable'][:25] for v in top_vars]
    p_values = [-np.log10(v['P_Value']) for v in top_vars]
    
    bars = ax.barh(range(len(var_names)), p_values, color='#666666')
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('-log10(p-value)')
    ax.set_title('Top Variables by Significance Level')
    ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=0.5, label='p=0.05 threshold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend()

# 2. Missingness overview pie chart
plot_count += 1
ax = plt.subplot(2, 3, plot_count)

missing_pct = (len(missing_data) / len(df)) * 100
complete_pct = 100 - missing_pct
colors = ['#CCCCCC', '#666666']

wedges, texts, autotexts = ax.pie([missing_pct, complete_pct], 
                                    labels=[f'Missing\n({missing_pct:.1f}%)', 
                                            f'Complete\n({complete_pct:.1f}%)'],
                                    colors=colors,
                                    autopct='%1.1f%%',
                                    startangle=90,
                                    textprops={'fontsize': 9})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title(f'{TARGET_VARIABLE} Missingness Overview')

# 3. Effect sizes for numeric variables
numeric_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
if numeric_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in numeric_sig[:10]]
    differences = [v['Difference'] for v in numeric_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in differences]
    ax.barh(range(len(var_names)), differences, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Mean Difference')
    ax.set_title('Numeric Variables: Mean Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 4. Categorical variable distribution differences
categorical_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
if categorical_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [v['Variable'][:20] for v in categorical_sig[:10]]
    diff_pct = [v['Difference_%'] if pd.notna(v['Difference_%']) else 0 for v in categorical_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in diff_pct]
    ax.barh(range(len(var_names)), diff_pct, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('Difference in Most Common Category (%)')
    ax.set_title('Categorical Variables: Distribution Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 5. Race variables detailed comparison
race_sig = [v for v in categorical_sig if 'Race' in v['Variable'] or 'race' in v['Variable']]
if race_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    race_data = []
    race_labels = []
    for var in race_sig[:2]:  # Show top 2 race variables
        # Get detailed distribution if available
        race_labels.append(var['Variable'])
        missing_pct = float(var['Statistic'].split('(')[1].split('%')[0])
        complete_pct = float(var['Comparison'].split('(')[1].split('%')[0])
        race_data.append([missing_pct, complete_pct])
    
    if race_data:
        x = np.arange(len(race_labels))
        width = 0.35
        
        ax.bar(x - width/2, [d[0] for d in race_data], width, label='Missing Group', color='#CC6666')
        ax.bar(x + width/2, [d[1] for d in race_data], width, label='Complete Group', color='#66CC66')
        
        ax.set_xlabel('Race Variables')
        ax.set_ylabel('Most Common Category Percentage (%)')
        ax.set_title('Race Variables: Missing vs Complete Group\nMost Common Category Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(race_labels, rotation=45, ha='right')
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

# 6. Summary text panel
plot_count += 1
ax = plt.subplot(2, 3, plot_count)
ax.axis('off')

sig_count = len(significant_vars_list)
total_vars = len(columns_to_analyze)
numeric_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Numeric'])
categorical_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Categorical'])

summary_text = f"IUPAC Statistical Summary\n\n"
summary_text += f"Target Variable: {TARGET_VARIABLE}\n"
summary_text += f"Missing rate: {missing_pct:.1f}%\n"
summary_text += f"Missing group: {len(missing_data):,} records\n"
summary_text += f"Complete group: {len(complete_data):,} records\n\n"
summary_text += f"Total variables analyzed: {total_vars}\n"
summary_text += f"Variables with significant differences: {sig_count}\n"
summary_text += f"  • Numeric: {numeric_sig_count}\n"
summary_text += f"  • Categorical: {categorical_sig_count}\n\n"

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN - Strong evidence of non-random missingness"
elif sig_count > 2:
    pattern = "MODERATE PATTERN - Some systematic patterns detected"
else:
    pattern = "RANDOM PATTERN - Missing appears relatively random"

summary_text += f"Missingness pattern: {pattern}\n\n"
summary_text += f"Included Race Variables: {len([v for v in categorical_sig if 'Race' in v['Variable']])} significant\n\n"
summary_text += f"Generated: {datetime.now().strftime('%Y-%m-%d')}"

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', family='sans-serif',
        bbox=dict(boxstyle='round', facecolor='#F0F0F0', edgecolor='#CCCCCC'))

plt.suptitle(f'Missing {TARGET_VARIABLE} Analysis\nIUPAC Compliant Statistical Report', 
             fontsize=12, fontweight='normal', y=1.02)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save with IUPAC specifications
output_png = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.png'
try:
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print(f"✅ Created IUPAC compliant visualization: {output_png}")
except Exception as e:
    print(f"❌ Error saving PNG: {e}")

# ============================================
# 4. CREATE IUPAC COMPLIANT WORD DOCUMENT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 4: CREATING IUPAC COMPLIANT WORD DOCUMENT")
print(f"{'='*80}")

# Create Word document
doc = Document()

# Set document properties
doc.core_properties.author = "Data Analysis Report"
doc.core_properties.created = datetime.now()

# Add title
title = doc.add_heading(f'Missing {TARGET_VARIABLE} Analysis Report', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add metadata
doc.add_paragraph(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
doc.add_paragraph(f'Dataset: {len(df):,} rows × {len(df.columns)} columns')
doc.add_paragraph(f'Target Variable: {TARGET_VARIABLE}')
doc.add_paragraph()

# Executive Summary
doc.add_heading('1. Executive Summary', level=1)

summary_text = f"""
This report presents a comprehensive IUPAC-compliant statistical analysis of missing values in {TARGET_VARIABLE}.

Key findings include:
• Missing records: {len(missing_data):,} ({missing_pct:.1f}% of total data)
• Variables with significant differences (p < 0.05): {sig_count}
  - Numeric variables: {numeric_sig_count}
  - Categorical variables: {categorical_sig_count}
  - Race variables: {len([v for v in significant_vars_list if 'Race' in v['Variable']])}
• Total variables analyzed: {total_vars}

Based on the statistical analysis, the missingness pattern is classified as:
"""
if sig_count > 5:
    summary_text += "\nSYSTEMATIC PATTERN - Missing data is strongly associated with other variables, suggesting non-random missingness."
elif sig_count > 2:
    summary_text += "\nMODERATE PATTERN - Some systematic patterns detected in the missing data."
else:
    summary_text += "\nRANDOM PATTERN - Missing data appears relatively random."

doc.add_paragraph(summary_text.strip())

# Statistical Summary
doc.add_heading('2. Statistical Summary', level=1)

# Create summary table
summary_table = doc.add_table(rows=len(summary_stats) + 1, cols=2)
summary_table.style = 'Light Grid Accent 1'
summary_table.autofit = False
summary_table.columns[0].width = Inches(4)
summary_table.columns[1].width = Inches(3)

# Header
header_cells = summary_table.rows[0].cells
header_cells[0].text = 'Metric'
header_cells[1].text = 'Value'
for cell in header_cells:
    cell.paragraphs[0].runs[0].font.bold = True
    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

# Data
for i, row in summary_stats.iterrows():
    cells = summary_table.rows[i + 1].cells
    cells[0].text = str(row['Metric'])
    cells[1].text = str(row['Value'])
    cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT

# ============================================
# DETAILED SIGNIFICANT DIFFERENCES SECTION
# ============================================

if significant_vars_list:
    doc.add_heading('3. Variables with Significant Differences (p < 0.05)', level=1)
    doc.add_paragraph(f'The following {len(significant_vars_list)} variables show statistically significant differences between missing and complete groups:')
    doc.add_paragraph()
    
    # Split into numeric and categorical
    numeric_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
    categorical_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
    
    # 3.1 Numeric Variables Section
    if numeric_vars_sig:
        doc.add_heading('3.1 Numeric Variables', level=2)
        
        # Create table for numeric variables
        num_table = doc.add_table(rows=len(numeric_vars_sig) + 1, cols=6)
        num_table.style = 'Light Grid Accent 1'
        num_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.2), Inches(1.2), Inches(1.2), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            num_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Mean', 'Complete Group Mean', 'Difference', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = num_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(numeric_vars_sig):
            row = num_table.rows[i + 1].cells
            row[0].text = var['Variable']
            
            # Extract mean from statistic string
            mean_val = var['Statistic'].split('=')[1].split()[0]
            row[1].text = mean_val
            
            # Extract complete mean
            comp_mean = var['Comparison'].split('=')[1]
            row[2].text = comp_mean
            
            row[3].text = f"{var['Difference']:+.2f}"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [1, 2, 3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.2 Categorical Variables Section (including race variables)
    if categorical_vars_sig:
        doc.add_heading('3.2 Categorical Variables', level=2)
        
        # Create table for categorical variables
        cat_table = doc.add_table(rows=len(categorical_vars_sig) + 1, cols=6)
        cat_table.style = 'Light Grid Accent 1'
        cat_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.8), Inches(1.5), Inches(1.5), Inches(1.0), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            cat_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group Distribution', 'Complete Group Distribution', 'Difference (%)', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = cat_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(categorical_vars_sig):
            row = cat_table.rows[i + 1].cells
            row[0].text = var['Variable']
            row[1].text = var['Statistic']
            row[2].text = var['Comparison']
            row[3].text = f"{var['Difference']:+.1f}%" if pd.notna(var['Difference']) else "N/A"
            row[4].text = f"{var['P_Value']:.4f}"
            row[5].text = var['Interpretation']
            
            # Right-align numeric columns
            for col_idx in [3, 4]:
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
        
        doc.add_paragraph()
    
    # 3.3 Detailed Race Variable Analysis
    race_vars_sig = [v for v in categorical_vars_sig if 'Race' in v['Variable'] or 'race' in v['Variable']]
    if race_vars_sig:
        doc.add_heading('3.3 Detailed Race Variable Analysis', level=2)
        
        for race_var in race_vars_sig:
            doc.add_heading(f'Race Variable: {race_var["Variable"]}', level=3)
            
            # Get detailed distribution for this race variable
            col_name = race_var['Variable']
            if col_name in missing_data.columns:
                missing_valid = missing_data[col_name].dropna()
                complete_valid = complete_data[col_name].dropna()
                
                # Create distribution comparison table
                all_categories = sorted(set(missing_valid.value_counts().index) | set(complete_valid.value_counts().index))
                
                # Create comparison table
                comparison_table = doc.add_table(rows=len(all_categories) + 1, cols=4)
                comparison_table.style = 'Light Grid Accent 1'
                comparison_table.autofit = False
                
                # Set column widths
                widths = [Inches(1.5), Inches(1.5), Inches(1.5), Inches(1.5)]
                for i, width in enumerate(widths):
                    comparison_table.columns[i].width = width
                
                # Headers
                headers = ['Category', 'Missing Group (%)', 'Complete Group (%)', 'Missing Rate (%)']
                for i, header in enumerate(headers):
                    cell = comparison_table.rows[0].cells[i]
                    cell.text = header
                    cell.paragraphs[0].runs[0].font.bold = True
                    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
                
                # Data
                for i, category in enumerate(all_categories):
                    row = comparison_table.rows[i + 1].cells
                    row[0].text = str(category)
                    
                    # Calculate percentages
                    missing_count = (missing_valid == category).sum()
                    complete_count = (complete_valid == category).sum()
                    total_count = missing_count + complete_count
                    
                    missing_pct = (missing_count / len(missing_valid)) * 100 if len(missing_valid) > 0 else 0
                    complete_pct = (complete_count / len(complete_valid)) * 100 if len(complete_valid) > 0 else 0
                    missing_rate = (missing_count / total_count) * 100 if total_count > 0 else 0
                    
                    row[1].text = f"{missing_pct:.1f}%"
                    row[2].text = f"{complete_pct:.1f}%"
                    row[3].text = f"{missing_rate:.1f}%"
                    
                    # Right-align numeric columns
                    for col_idx in [1, 2, 3]:
                        row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT
                
                doc.add_paragraph()
                
                # Add interpretation
                doc.add_paragraph(f"Interpretation: {race_var['Interpretation']}")
                doc.add_paragraph()
    
    # 3.4 Summary of Key Findings
    doc.add_heading('3.4 Summary of Key Findings', level=2)
    
    findings_text = f"""
The analysis identified {len(significant_vars_list)} variables with statistically significant differences (p < 0.05):
• {len(numeric_vars_sig)} numeric variables show significant mean differences
• {len(categorical_vars_sig)} categorical variables show significant distribution differences
• {len(race_vars_sig)} race-related variables show significant differences

Top 10 Most Significant Variables:
"""
    
    # Add top 10 most significant findings
    sorted_vars = sorted(significant_vars_list, key=lambda x: x['P_Value'])[:10]
    for i, var in enumerate(sorted_vars, 1):
        if var['Type'] == 'Numeric':
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
        else:
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
    
    doc.add_paragraph(findings_text.strip())

# Add visualization
doc.add_heading('4. Statistical Visualizations', level=1)
if os.path.exists(output_png):
    doc.add_picture(output_png, width=Inches(6.5))
    doc.add_paragraph(f'Figure 1. IUPAC compliant statistical visualization of missing {TARGET_VARIABLE} patterns.')

# Recommendations
doc.add_heading('5. Recommendations', level=1)

if sig_count > 5:
    doc.add_paragraph('Based on the statistical analysis, the missing data exhibits SYSTEMATIC patterns:', style='List Bullet')
    doc.add_paragraph('Consider treating missing as a separate category in analysis', style='List Bullet')
    doc.add_paragraph('Use multiple imputation methods (MICE, EM algorithm)', style='List Bullet')
    doc.add_paragraph('Investigate the following key variables for common themes:', style='List Bullet')
    
    # Add list of significant variables
    for var in significant_vars_list[:10]:
        doc.add_paragraph(f'  - {var["Variable"]} (p={var["P_Value"]:.4f})', style='List Bullet')
    
    doc.add_paragraph('Document missingness patterns in methodology section', style='List Bullet')
    
elif sig_count > 2:
    doc.add_paragraph('Based on the statistical analysis, the missing data shows MODERATE systematic patterns:', style='List Bullet')
    doc.add_paragraph('Consider multiple imputation or sensitivity analysis', style='List Bullet')
    doc.add_paragraph('Document the patterns in your methodology', style='List Bullet')
    doc.add_paragraph('Validate findings with domain experts', style='List Bullet')
    
else:
    doc.add_paragraph('Based on the statistical analysis, the missing data appears RELATIVELY RANDOM:', style='List Bullet')
    doc.add_paragraph('Simple imputation methods may be acceptable (mean, median, mode)', style='List Bullet')
    doc.add_paragraph('Listwise deletion may be appropriate if missing rate is low', style='List Bullet')
    doc.add_paragraph('Still verify randomness with domain knowledge', style='List Bullet')

# Methodology
doc.add_heading('6. Methodology (IUPAC Compliant)', level=1)
doc.add_paragraph('Analysis performed according to IUPAC guidelines for data presentation:')
doc.add_paragraph('• Missing values reported as both count (n) and percentage (%)', style='List Bullet')
doc.add_paragraph('• Tables formatted with thin borders and alternating row shading', style='List Bullet')
doc.add_paragraph('• Sans-serif fonts (Arial) used for optimal readability', style='List Bullet')
doc.add_paragraph('• Statistical tests: t-test (numeric), χ² test (categorical)', style='List Bullet')
doc.add_paragraph('• Significance threshold: α = 0.05', style='List Bullet')
doc.add_paragraph('• Effect sizes reported where applicable', style='List Bullet')
doc.add_paragraph('• Figures saved at 300 DPI for publication quality', style='List Bullet')

# Add footer with page numbers
section = doc.sections[0]
footer = section.footer
footer_para = footer.paragraphs[0]
footer_para.text = f"{TARGET_VARIABLE} Missing Analysis | Generated: {datetime.now().strftime('%Y-%m-%d')} | Page "
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add page number field
run = footer_para.add_run()
fldChar1 = OxmlElement('w:fldChar')
fldChar1.set(qn('w:fldCharType'), 'begin')
run._r.append(fldChar1)

instrText = OxmlElement('w:instrText')
instrText.text = "PAGE"
run._r.append(instrText)

fldChar2 = OxmlElement('w:fldChar')
fldChar2.set(qn('w:fldCharType'), 'end')
run._r.append(fldChar2)

# Save document
output_docx = f'missing_{safe_filename_base}_iupac_report_{timestamp}.docx'
try:
    doc.save(output_docx)
    print(f"✅ Created IUPAC compliant Word document: {output_docx}")
except PermissionError as e:
    alt_filename = f'missing_{safe_filename_base}_iupac_report_{timestamp}_new.docx'
    doc.save(alt_filename)
    print(f"✅ Saved with alternative filename: {alt_filename}")
    output_docx = alt_filename

# ============================================
# 5. FINAL SUMMARY
# ============================================

print(f"\n{'='*80}")
print(f"📋 IUPAC COMPLIANT ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📊 FILES GENERATED (IUPAC Compliant):")
print(f"   • Excel Report: {output_file}")
print(f"   • Visualization: {output_png}")
print(f"   • Word Report: {output_docx}")

print(f"\n📈 STATISTICAL SUMMARY:")
print(f"   • Missing {TARGET_VARIABLE}: {len(missing_data):,} ({missing_pct:.1f}%)")
print(f"   • Variables with significant differences: {sig_count}")
print(f"     - Numeric: {numeric_sig_count}")
print(f"     - Categorical: {categorical_sig_count}")
print(f"     - Race variables: {len([v for v in significant_vars_list if 'Race' in v['Variable']])}")

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN"
elif sig_count > 2:
    pattern = "MODERATE PATTERN"
else:
    pattern = "RANDOM PATTERN"

print(f"   • Missingness pattern: {pattern}")

print(f"\n📋 TOP SIGNIFICANT VARIABLES:")
for var in significant_vars_list[:10]:
    print(f"   • {var['Variable']}: {var['Statistic']}, p={var['P_Value']:.4f}")

print(f"\n{'='*80}")
print(f"✅ All IUPAC compliant files generated successfully")
print(f"{'='*80}")


📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING Multiple_Birth_Status PATTERNS
(IUPAC Compliant Output)

📌 DATASET OVERVIEW:
   • Total records with missing Multiple_Birth_Status: 363,824
   • Total records with complete Multiple_Birth_Status: 57
   • Missing percentage: 99.98%

📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE

Analyzing 17 variables...

✅ Analysis complete. Found 8 variables with significant differences (p < 0.05)

📋 SIGNIFICANT VARIABLES (p < 0.05):
   • Registered_Month: Most common='October' (8.9%), p=0.0000
   • Registered_District_2.0: Mean=41.60, p=0.0329
   • Registered_District: Most common='Colombo' (15.2%), p=0.0000
   • Birh_Year: Mean=2009.57, p=0.0000
   • Birth_Month: Most common='October' (9.3%), p=0.0038
   • Birth_Weight(grams): Mean=2887.63, p=0.0000
   • Birth_Order: Most common='First' (45.9%), p=0.0020
   • District_of_Mother: Most common='Colombo' (10.3%), p=0.0000

📌 ADDITIONAL ANALYSIS: DETAILED CATEGORICAL BREAKDOWN

📊 De

In [10]:
# ============================================
# FOCUS ON BIRTH WEIGHT - ANALYZE MISSING PATTERNS
# ============================================
TARGET_VARIABLE = 'Multiple_Birth_Status'  # <-- Target variable to analyze
# ============================================

# ============================================
# COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING BIRTH WEIGHT PATTERNS
# WITH IUPAC COMPLIANT OUTPUT (Excel, Word, PNG)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
import os

warnings.filterwarnings('ignore')

# Create timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n{'='*80}")
print(f"📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING {TARGET_VARIABLE} PATTERNS")
print(f"(IUPAC Compliant Output)")
print(f"{'='*80}")

# Load the missing data
missing_data = df[df[TARGET_VARIABLE].isnull()].copy()
complete_data = df[df[TARGET_VARIABLE].notnull()].copy()

print(f"\n📌 DATASET OVERVIEW:")
print(f"   • Total records with missing {TARGET_VARIABLE}: {len(missing_data):,}")
print(f"   • Total records with complete {TARGET_VARIABLE}: {len(complete_data):,}")
print(f"   • Missing percentage: {(len(missing_data) / len(df)) * 100:.2f}%")

# ============================================
# 1. UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE")
print(f"{'='*80}")

# Create a comprehensive statistics dataframe
all_stats = []
significant_vars_list = []  # Store significant variables for Word document

# List all columns to analyze (exclude target variable)
columns_to_analyze = [col for col in missing_data.columns if col != TARGET_VARIABLE]

print(f"\nAnalyzing {len(columns_to_analyze)} variables...")

# Analyze each column
for col in columns_to_analyze:
    col_stats = {
        'Variable': col,
        'Data_Type': str(missing_data[col].dtype),
        'Missing_in_Group': missing_data[col].isnull().sum(),
        'Missing_in_Group_%': (missing_data[col].isnull().sum() / len(missing_data)) * 100,
        'Complete_in_Group': missing_data[col].notnull().sum(),
        'Complete_in_Group_%': (missing_data[col].notnull().sum() / len(missing_data)) * 100,
        'Unique_Values': missing_data[col].nunique(),
    }
    
    # Check if variable is numeric
    if missing_data[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(missing_data[col]):
        # NUMERIC VARIABLE STATISTICS
        valid_data = missing_data[col].dropna()
        complete_valid_data = complete_data[col].dropna() if col in complete_data.columns else pd.Series()
        
        if len(valid_data) > 0:
            if valid_data.nunique() > 1:
                try:
                    col_stats.update({
                        'Mean_Missing': valid_data.mean(),
                        'Median_Missing': valid_data.median(),
                        'Std_Dev_Missing': valid_data.std(),
                        'Min_Missing': valid_data.min(),
                        'Max_Missing': valid_data.max(),
                        'Count_Missing': len(valid_data),
                        'Count_Complete': len(complete_valid_data)
                    })
                except:
                    col_stats.update({
                        'Mean_Missing': np.nan, 'Median_Missing': np.nan, 
                        'Std_Dev_Missing': np.nan, 'Min_Missing': np.nan, 'Max_Missing': np.nan,
                        'Count_Missing': 0, 'Count_Complete': 0
                    })
            else:
                constant_value = valid_data.iloc[0] if len(valid_data) > 0 else np.nan
                col_stats.update({
                    'Mean_Missing': constant_value, 'Median_Missing': constant_value, 
                    'Std_Dev_Missing': 0, 'Min_Missing': constant_value, 'Max_Missing': constant_value,
                    'Count_Missing': len(valid_data), 'Count_Complete': len(complete_valid_data)
                })
            
            # Compare with complete data
            if len(complete_valid_data) > 0:
                if len(valid_data) > 1 and len(complete_valid_data) > 1:
                    try:
                        t_stat, p_value = stats.ttest_ind(valid_data, complete_valid_data, equal_var=False)
                        col_stats['P_Value'] = p_value
                        col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                        col_stats['Mean_Difference'] = valid_data.mean() - complete_valid_data.mean()
                        
                        # Add complete group statistics
                        col_stats['Mean_Complete'] = complete_valid_data.mean()
                        col_stats['Median_Complete'] = complete_valid_data.median()
                        
                        # Store for detailed report with counts
                        if p_value < 0.05:
                            significant_vars_list.append({
                                'Variable': col,
                                'Type': 'Numeric',
                                'Statistic': f"Mean={valid_data.mean():.2f} (n={len(valid_data):,})",
                                'Comparison': f"Complete Mean={complete_valid_data.mean():.2f} (n={len(complete_valid_data):,})",
                                'Difference': col_stats['Mean_Difference'],
                                'Difference_%': (col_stats['Mean_Difference'] / abs(complete_valid_data.mean())) * 100 if complete_valid_data.mean() != 0 else np.nan,
                                'P_Value': p_value,
                                'Count_Missing': len(valid_data),
                                'Count_Complete': len(complete_valid_data),
                                'Interpretation': f"Missing group (n={len(valid_data):,}) has {abs(col_stats['Mean_Difference']):.2f} units {'higher' if col_stats['Mean_Difference'] > 0 else 'lower'} than complete group (n={len(complete_valid_data):,})"
                            })
                    except:
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'NO'
                        col_stats['Mean_Difference'] = np.nan
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    col_stats['Mean_Difference'] = np.nan
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
                col_stats['Mean_Difference'] = np.nan
        else:
            col_stats.update({
                'Mean_Missing': np.nan, 'Median_Missing': np.nan, 'Std_Dev_Missing': np.nan,
                'Min_Missing': np.nan, 'Max_Missing': np.nan, 'P_Value': np.nan,
                'Significant_Difference': 'N/A', 'Mean_Difference': np.nan,
                'Count_Missing': 0, 'Count_Complete': 0
            })
            
    else:
        # CATEGORICAL VARIABLE STATISTICS (including race variables)
        valid_data = missing_data[col].dropna()
        
        if len(valid_data) > 0:
            value_counts = valid_data.value_counts()
            top_categories = value_counts.head(5)
            
            col_stats.update({
                'Most_Common': str(top_categories.index[0]) if len(top_categories) > 0 else np.nan,
                'Most_Common_Count': top_categories.iloc[0] if len(top_categories) > 0 else 0,
                'Most_Common_%': (top_categories.iloc[0] / len(valid_data)) * 100 if len(top_categories) > 0 else 0,
                '2nd_Common': str(top_categories.index[1]) if len(top_categories) > 1 else np.nan,
                '2nd_Common_Count': top_categories.iloc[1] if len(top_categories) > 1 else 0,
                '2nd_Common_%': (top_categories.iloc[1] / len(valid_data)) * 100 if len(top_categories) > 1 else 0,
                '3rd_Common': str(top_categories.index[2]) if len(top_categories) > 2 else np.nan,
                '3rd_Common_Count': top_categories.iloc[2] if len(top_categories) > 2 else 0,
                '3rd_Common_%': (top_categories.iloc[2] / len(valid_data)) * 100 if len(top_categories) > 2 else 0,
                'Total_Valid_Missing': len(valid_data),
            })
            
            # Chi-square test comparing with complete data
            if col in complete_data.columns:
                complete_valid = complete_data[col].dropna()
                if len(complete_valid) > 0:
                    try:
                        missing_cats = valid_data.value_counts()
                        complete_cats = complete_valid.value_counts()
                        
                        # Get all categories
                        all_cats = sorted(set(missing_cats.index) | set(complete_cats.index))
                        missing_counts = [missing_cats.get(cat, 0) for cat in all_cats]
                        complete_counts = [complete_cats.get(cat, 0) for cat in all_cats]
                        
                        # Only run chi-square if we have enough data and at least 2 categories
                        if len(all_cats) > 1 and sum(missing_counts) > 0 and sum(complete_counts) > 0:
                            # Check if any expected frequencies are < 5 (if so, use Fisher's exact or report warning)
                            contingency = np.array([missing_counts, complete_counts])
                            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
                            
                            col_stats['P_Value'] = p_value
                            col_stats['Significant_Difference'] = 'YES' if p_value < 0.05 else 'NO'
                            
                            # Add complete group distribution with counts
                            complete_top = complete_cats.index[0] if len(complete_cats) > 0 else 'N/A'
                            complete_top_count = complete_cats.iloc[0] if len(complete_cats) > 0 else 0
                            complete_top_pct = (complete_top_count / len(complete_valid)) * 100 if len(complete_cats) > 0 else 0
                            col_stats['Most_Common_Complete'] = complete_top
                            col_stats['Most_Common_Complete_Count'] = complete_top_count
                            col_stats['Most_Common_Complete_%'] = complete_top_pct
                            col_stats['Total_Valid_Complete'] = len(complete_valid)
                            
                            # Calculate additional metrics for categorical variables
                            missing_pct_by_cat = {}
                            for cat in all_cats:
                                missing_count = missing_cats.get(cat, 0)
                                complete_count = complete_cats.get(cat, 0)
                                total = missing_count + complete_count
                                if total > 0:
                                    missing_pct_by_cat[cat] = (missing_count / total) * 100
                            
                            col_stats['Missing_Rate_by_Category'] = missing_pct_by_cat
                            
                            # Store for detailed report with counts
                            if p_value < 0.05:
                                # Calculate the difference in proportions for the most common category
                                diff_pct = col_stats['Most_Common_%'] - complete_top_pct
                                
                                significant_vars_list.append({
                                    'Variable': col,
                                    'Type': 'Categorical',
                                    'Statistic': f"Most common='{col_stats['Most_Common']}' ({col_stats['Most_Common_Count']:,}/{len(valid_data):,}, {col_stats['Most_Common_%']:.1f}%)",
                                    'Comparison': f"Complete: Most common='{complete_top}' ({complete_top_count:,}/{len(complete_valid):,}, {complete_top_pct:.1f}%)",
                                    'Difference': diff_pct,
                                    'Difference_%': (diff_pct / complete_top_pct * 100) if complete_top_pct != 0 else np.nan,
                                    'P_Value': p_value,
                                    'Count_Missing_Total': len(valid_data),
                                    'Count_Complete_Total': len(complete_valid),
                                    'Count_Missing_Category': col_stats['Most_Common_Count'],
                                    'Count_Complete_Category': complete_top_count,
                                    'Interpretation': f"Missing group (n={len(valid_data):,}) shows different distribution (p={p_value:.4f})",
                                    'Chi2_Statistic': chi2,
                                    'Degrees_of_Freedom': dof
                                })
                        else:
                            col_stats['P_Value'] = np.nan
                            col_stats['Significant_Difference'] = 'INSUFFICIENT_DATA'
                    except Exception as e:
                        print(f"   Warning: Error analyzing {col}: {e}")
                        col_stats['P_Value'] = np.nan
                        col_stats['Significant_Difference'] = 'ERROR'
                else:
                    col_stats['P_Value'] = np.nan
                    col_stats['Significant_Difference'] = 'N/A'
            else:
                col_stats['P_Value'] = np.nan
                col_stats['Significant_Difference'] = 'N/A'
        else:
            col_stats.update({
                'Most_Common': np.nan, 'Most_Common_Count': 0, 'Most_Common_%': 0,
                '2nd_Common': np.nan, '2nd_Common_Count': 0, '2nd_Common_%': 0,
                '3rd_Common': np.nan, '3rd_Common_Count': 0, '3rd_Common_%': 0,
                'P_Value': np.nan, 'Significant_Difference': 'N/A',
                'Total_Valid_Missing': 0
            })
    
    all_stats.append(col_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_stats)

# Sort by significance
stats_df = stats_df.sort_values('P_Value', ascending=True)

print(f"\n✅ Analysis complete. Found {len(significant_vars_list)} variables with significant differences (p < 0.05)")

# Print significant variables in console with counts
if significant_vars_list:
    print(f"\n📋 SIGNIFICANT VARIABLES (p < 0.05):")
    for var in significant_vars_list:
        if var['Type'] == 'Numeric':
            print(f"   • {var['Variable']}: {var['Statistic']} vs {var['Comparison']}, p={var['P_Value']:.4f}")
        else:
            print(f"   • {var['Variable']}: {var['Statistic']} vs {var['Comparison']}, p={var['P_Value']:.4f}")

# ============================================
# ADDITIONAL ANALYSIS: DETAILED CATEGORICAL ANALYSIS FOR RACE VARIABLES
# ============================================

print(f"\n{'='*80}")
print(f"📌 ADDITIONAL ANALYSIS: DETAILED CATEGORICAL BREAKDOWN")
print(f"{'='*80}")

# Specifically analyze race variables with more detail
race_vars = [col for col in columns_to_analyze if 'Race' in col or 'race' in col]
for race_var in race_vars:
    if race_var in missing_data.columns:
        print(f"\n📊 Detailed Analysis for {race_var}:")
        
        missing_valid = missing_data[race_var].dropna()
        complete_valid = complete_data[race_var].dropna()
        
        if len(missing_valid) > 0 and len(complete_valid) > 0:
            # Get distributions with counts
            missing_dist = missing_valid.value_counts()
            complete_dist = complete_valid.value_counts()
            
            print(f"\n   Distribution in MISSING group (n={len(missing_valid):,} total):")
            for category, count in missing_dist.head(5).items():
                pct = (count / len(missing_valid)) * 100
                print(f"      • {category}: {count:,}/{len(missing_valid):,} ({pct:.1f}%)")
            
            print(f"\n   Distribution in COMPLETE group (n={len(complete_valid):,} total):")
            for category, count in complete_dist.head(5).items():
                pct = (count / len(complete_valid)) * 100
                print(f"      • {category}: {count:,}/{len(complete_valid):,} ({pct:.1f}%)")
            
            # Calculate missing rate for each category with counts
            all_categories = sorted(set(missing_dist.index) | set(complete_dist.index))
            print(f"\n   Missing rate by category:")
            for category in all_categories:
                missing_count = (missing_data[race_var] == category).sum()
                complete_count = (complete_data[race_var] == category).sum()
                total = missing_count + complete_count
                if total > 0:
                    missing_rate = (missing_count / total) * 100
                    print(f"      • {category}: {missing_count:,}/{total:,} ({missing_rate:.1f}%)")

# ============================================
# 2. CREATE IUPAC COMPLIANT EXCEL REPORT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 2: CREATING IUPAC COMPLIANT EXCEL REPORT")
print(f"{'='*80}")

safe_filename_base = TARGET_VARIABLE.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
output_file = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.xlsx'

try:
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # Sheet 1: Variable-level Statistics
        stats_df.to_excel(writer, sheet_name='Variable_Statistics', index=False)
        
        # Sheet 2: Summary Statistics
        numeric_vars = [col for col in columns_to_analyze if missing_data[col].dtype in ['int64', 'float64']]
        categorical_vars = [col for col in columns_to_analyze if missing_data[col].dtype not in ['int64', 'float64']]
        
        summary_stats = pd.DataFrame({
            'Metric': [
                f'Total Missing {TARGET_VARIABLE} Records',
                f'Missing Percentage',
                'Total Variables Analyzed',
                'Numeric Variables',
                'Categorical Variables',
                'Variables with Significant Differences (p < 0.05)',
                'Analysis Date'
            ],
            'Value': [
                f"{len(missing_data):,}",
                f"{(len(missing_data) / len(df)) * 100:.2f}%",
                len(columns_to_analyze),
                len(numeric_vars),
                len(categorical_vars),
                len(significant_vars_list),
                datetime.now().strftime("%Y-%m-%d")
            ]
        })
        summary_stats.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 3: Variables with Significant Differences (Detailed)
        if significant_vars_list:
            sig_diff_df = pd.DataFrame(significant_vars_list)
            sig_diff_df.to_excel(writer, sheet_name='Significant_Differences_Detailed', index=False)
        
        # Format Excel sheets with IUPAC styling
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Auto-fit columns
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Style header row
            header_font = Font(name='Arial', size=10, bold=False, color='000000')
            header_fill = PatternFill(start_color='F0F0F0', end_color='F0F0F0', fill_type='solid')
            header_alignment = Alignment(horizontal='center', vertical='center')
            
            for cell in worksheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
            
            # Style data cells
            thin_border = Border(
                left=Side(style='thin', color='CCCCCC'),
                right=Side(style='thin', color='CCCCCC'),
                top=Side(style='thin', color='CCCCCC'),
                bottom=Side(style='thin', color='CCCCCC')
            )
            
            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name='Arial', size=9)
                    cell.border = thin_border
                    if isinstance(cell.value, (int, float)):
                        cell.alignment = Alignment(horizontal='right')
                    else:
                        cell.alignment = Alignment(horizontal='left')
    
    print(f"✅ Created IUPAC compliant Excel report: {output_file}")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

# ============================================
# 3. CREATE IUPAC COMPLIANT VISUALIZATIONS
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 3: CREATING IUPAC COMPLIANT VISUALIZATIONS")
print(f"{'='*80}")

# Set IUPAC style for matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8

# Determine number of plots needed
num_plots = min(6, 2 + len(significant_vars_list) // 5)
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('white')
plot_count = 0

# 1. Bar plot for significant differences
if significant_vars_list:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    # Get top 15 significant variables
    top_vars = significant_vars_list[:15]
    var_names = [v['Variable'][:25] for v in top_vars]
    p_values = [-np.log10(v['P_Value']) for v in top_vars]
    
    bars = ax.barh(range(len(var_names)), p_values, color='#666666')
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=8)
    ax.set_xlabel('-log10(p-value)')
    ax.set_title('Top Variables by Significance Level')
    ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=0.5, label='p=0.05 threshold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend()

# 2. Missingness overview pie chart
plot_count += 1
ax = plt.subplot(2, 3, plot_count)

missing_pct = (len(missing_data) / len(df)) * 100
complete_pct = 100 - missing_pct
colors = ['#CCCCCC', '#666666']

wedges, texts, autotexts = ax.pie([missing_pct, complete_pct], 
                                    labels=[f'Missing\n(n={len(missing_data):,}, {missing_pct:.1f}%)', 
                                            f'Complete\n(n={len(complete_data):,}, {complete_pct:.1f}%)'],
                                    colors=colors,
                                    autopct='%1.1f%%',
                                    startangle=90,
                                    textprops={'fontsize': 9})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title(f'{TARGET_VARIABLE} Missingness Overview')

# 3. Effect sizes for numeric variables
numeric_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
if numeric_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [f"{v['Variable']}\n(n={v['Count_Missing']:,})" for v in numeric_sig[:10]]
    differences = [v['Difference'] for v in numeric_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in differences]
    ax.barh(range(len(var_names)), differences, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=7)
    ax.set_xlabel('Mean Difference')
    ax.set_title('Numeric Variables: Mean Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 4. Categorical variable distribution differences
categorical_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
if categorical_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    var_names = [f"{v['Variable']}\n(n={v['Count_Missing_Total']:,})" for v in categorical_sig[:10]]
    diff_pct = [v['Difference_%'] if pd.notna(v['Difference_%']) else 0 for v in categorical_sig[:10]]
    
    colors_diff = ['#CC6666' if x > 0 else '#66CC66' for x in diff_pct]
    ax.barh(range(len(var_names)), diff_pct, color=colors_diff)
    ax.set_yticks(range(len(var_names)))
    ax.set_yticklabels(var_names, fontsize=7)
    ax.set_xlabel('Difference in Most Common Category (%)')
    ax.set_title('Categorical Variables: Distribution Differences\n(Missing vs Complete)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 5. Race variables detailed comparison
race_sig = [v for v in categorical_sig if 'Race' in v['Variable'] or 'race' in v['Variable']]
if race_sig and plot_count < 6:
    plot_count += 1
    ax = plt.subplot(2, 3, plot_count)
    
    race_data = []
    race_labels = []
    for var in race_sig[:2]:  # Show top 2 race variables
        race_labels.append(f"{var['Variable']}\n(n={var['Count_Missing_Total']:,})")
        missing_pct = float(var['Statistic'].split('(')[-1].split('%')[0])
        complete_pct = float(var['Comparison'].split('(')[-1].split('%')[0])
        race_data.append([missing_pct, complete_pct])
    
    if race_data:
        x = np.arange(len(race_labels))
        width = 0.35
        
        ax.bar(x - width/2, [d[0] for d in race_data], width, label='Missing Group', color='#CC6666')
        ax.bar(x + width/2, [d[1] for d in race_data], width, label='Complete Group', color='#66CC66')
        
        ax.set_xlabel('Race Variables')
        ax.set_ylabel('Most Common Category Percentage (%)')
        ax.set_title('Race Variables: Missing vs Complete Group\nMost Common Category Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(race_labels, rotation=45, ha='right', fontsize=7)
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

# 6. Summary text panel
plot_count += 1
ax = plt.subplot(2, 3, plot_count)
ax.axis('off')

sig_count = len(significant_vars_list)
total_vars = len(columns_to_analyze)
numeric_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Numeric'])
categorical_sig_count = len([v for v in significant_vars_list if v['Type'] == 'Categorical'])

summary_text = f"IUPAC Statistical Summary\n\n"
summary_text += f"Target Variable: {TARGET_VARIABLE}\n"
summary_text += f"Missing rate: {missing_pct:.1f}%\n"
summary_text += f"Missing group: {len(missing_data):,} records\n"
summary_text += f"Complete group: {len(complete_data):,} records\n\n"
summary_text += f"Total variables analyzed: {total_vars}\n"
summary_text += f"Variables with significant differences: {sig_count}\n"
summary_text += f"  • Numeric: {numeric_sig_count}\n"
summary_text += f"  • Categorical: {categorical_sig_count}\n\n"

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN - Strong evidence of non-random missingness"
elif sig_count > 2:
    pattern = "MODERATE PATTERN - Some systematic patterns detected"
else:
    pattern = "RANDOM PATTERN - Missing appears relatively random"

summary_text += f"Missingness pattern: {pattern}\n\n"
summary_text += f"Included Race Variables: {len([v for v in categorical_sig if 'Race' in v['Variable']])} significant\n\n"
summary_text += f"Generated: {datetime.now().strftime('%Y-%m-%d')}"

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', family='sans-serif',
        bbox=dict(boxstyle='round', facecolor='#F0F0F0', edgecolor='#CCCCCC'))

plt.suptitle(f'Missing {TARGET_VARIABLE} Analysis\nIUPAC Compliant Statistical Report', 
             fontsize=12, fontweight='normal', y=1.02)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Save with IUPAC specifications
output_png = f'missing_{safe_filename_base}_iupac_stats_{timestamp}.png'
try:
    plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print(f"✅ Created IUPAC compliant visualization: {output_png}")
except Exception as e:
    print(f"❌ Error saving PNG: {e}")

# ============================================
# 4. CREATE IUPAC COMPLIANT WORD DOCUMENT
# ============================================

print(f"\n{'='*80}")
print(f"📌 PART 4: CREATING IUPAC COMPLIANT WORD DOCUMENT")
print(f"{'='*80}")

# Create Word document
doc = Document()

# Set document properties
doc.core_properties.author = "Data Analysis Report"
doc.core_properties.created = datetime.now()

# Add title
title = doc.add_heading(f'Missing {TARGET_VARIABLE} Analysis Report', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add metadata
doc.add_paragraph(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
doc.add_paragraph(f'Dataset: {len(df):,} rows × {len(df.columns)} columns')
doc.add_paragraph(f'Target Variable: {TARGET_VARIABLE}')
doc.add_paragraph()

# Executive Summary
doc.add_heading('1. Executive Summary', level=1)

summary_text = f"""
This report presents a comprehensive IUPAC-compliant statistical analysis of missing values in {TARGET_VARIABLE}.

Key findings include:
• Missing records: {len(missing_data):,} ({missing_pct:.1f}% of total data)
• Variables with significant differences (p < 0.05): {sig_count}
  - Numeric variables: {numeric_sig_count}
  - Categorical variables: {categorical_sig_count}
  - Race variables: {len([v for v in significant_vars_list if 'Race' in v['Variable']])}
• Total variables analyzed: {total_vars}

Based on the statistical analysis, the missingness pattern is classified as:
"""
if sig_count > 5:
    summary_text += "\nSYSTEMATIC PATTERN - Missing data is strongly associated with other variables, suggesting non-random missingness."
elif sig_count > 2:
    summary_text += "\nMODERATE PATTERN - Some systematic patterns detected in the missing data."
else:
    summary_text += "\nRANDOM PATTERN - Missing data appears relatively random."

doc.add_paragraph(summary_text.strip())

# Statistical Summary
doc.add_heading('2. Statistical Summary', level=1)

# Create summary table
summary_table = doc.add_table(rows=len(summary_stats) + 1, cols=2)
summary_table.style = 'Light Grid Accent 1'
summary_table.autofit = False
summary_table.columns[0].width = Inches(4)
summary_table.columns[1].width = Inches(3)

# Header
header_cells = summary_table.rows[0].cells
header_cells[0].text = 'Metric'
header_cells[1].text = 'Value'
for cell in header_cells:
    cell.paragraphs[0].runs[0].font.bold = True
    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

# Data
for i, row in summary_stats.iterrows():
    cells = summary_table.rows[i + 1].cells
    cells[0].text = str(row['Metric'])
    cells[1].text = str(row['Value'])
    cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT

# ============================================
# DETAILED SIGNIFICANT DIFFERENCES SECTION
# ============================================

if significant_vars_list:
    doc.add_heading('3. Variables with Significant Differences (p < 0.05)', level=1)
    doc.add_paragraph(f'The following {len(significant_vars_list)} variables show statistically significant differences between missing and complete groups:')
    doc.add_paragraph()
    
    # Split into numeric and categorical
    numeric_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Numeric']
    categorical_vars_sig = [v for v in significant_vars_list if v['Type'] == 'Categorical']
    
    # 3.1 Numeric Variables Section
    if numeric_vars_sig:
        doc.add_heading('3.1 Numeric Variables', level=2)
        
        # Create table for numeric variables
        num_table = doc.add_table(rows=len(numeric_vars_sig) + 1, cols=7)
        num_table.style = 'Light Grid Accent 1'
        num_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.5), Inches(1.2), Inches(1.2), Inches(1.2), Inches(0.8), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            num_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group (n)', 'Complete Group (n)', 'Mean Difference', 'Difference %', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = num_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(numeric_vars_sig):
            row = num_table.rows[i + 1].cells
            row[0].text = var['Variable']
            
            # Extract mean and n from statistic string
            mean_val = var['Statistic'].split('=')[1].split()[0]
            n_val = var['Statistic'].split('n=')[1].split(')')[0]
            row[1].text = f"{mean_val}\n(n={n_val})"
            
            # Extract complete mean and n
            comp_mean = var['Comparison'].split('=')[1].split()[0]
            comp_n = var['Comparison'].split('n=')[1].split(')')[0]
            row[2].text = f"{comp_mean}\n(n={comp_n})"
            
            row[3].text = f"{var['Difference']:+.2f}"
            row[4].text = f"{var['Difference_%']:+.1f}%" if pd.notna(var['Difference_%']) else "N/A"
            row[5].text = f"{var['P_Value']:.4f}"
            row[6].text = var['Interpretation']
            
            # Center align all cells
            for col_idx in range(7):
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        doc.add_paragraph()
    
    # 3.2 Categorical Variables Section (including race variables)
    if categorical_vars_sig:
        doc.add_heading('3.2 Categorical Variables', level=2)
        
        # Create table for categorical variables
        cat_table = doc.add_table(rows=len(categorical_vars_sig) + 1, cols=7)
        cat_table.style = 'Light Grid Accent 1'
        cat_table.autofit = False
        
        # Set column widths
        widths = [Inches(1.5), Inches(1.5), Inches(1.5), Inches(1.0), Inches(1.0), Inches(1.0), Inches(2.0)]
        for i, width in enumerate(widths):
            cat_table.columns[i].width = width
        
        # Headers
        headers = ['Variable', 'Missing Group\n(Category, n, %)', 'Complete Group\n(Category, n, %)', 'Difference (%)', 'Missing n (total)', 'p-value', 'Interpretation']
        for i, header in enumerate(headers):
            cell = cat_table.rows[0].cells[i]
            cell.text = header
            cell.paragraphs[0].runs[0].font.bold = True
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Data
        for i, var in enumerate(categorical_vars_sig):
            row = cat_table.rows[i + 1].cells
            row[0].text = var['Variable']
            row[1].text = var['Statistic']
            row[2].text = var['Comparison']
            row[3].text = f"{var['Difference']:+.1f}%" if pd.notna(var['Difference']) else "N/A"
            row[4].text = f"{var['Count_Missing_Total']:,}" if 'Count_Missing_Total' in var else "N/A"
            row[5].text = f"{var['P_Value']:.4f}"
            row[6].text = var['Interpretation']
            
            # Center align all cells
            for col_idx in range(7):
                row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        doc.add_paragraph()
    
    # 3.3 Detailed Race Variable Analysis
    race_vars_sig = [v for v in categorical_vars_sig if 'Race' in v['Variable'] or 'race' in v['Variable']]
    if race_vars_sig:
        doc.add_heading('3.3 Detailed Race Variable Analysis', level=2)
        
        for race_var in race_vars_sig:
            doc.add_heading(f'Race Variable: {race_var["Variable"]}', level=3)
            
            # Get detailed distribution for this race variable
            col_name = race_var['Variable']
            if col_name in missing_data.columns:
                missing_valid = missing_data[col_name].dropna()
                complete_valid = complete_data[col_name].dropna()
                
                # Create distribution comparison table
                all_categories = sorted(set(missing_valid.value_counts().index) | set(complete_valid.value_counts().index))
                
                # Create comparison table
                comparison_table = doc.add_table(rows=len(all_categories) + 1, cols=5)
                comparison_table.style = 'Light Grid Accent 1'
                comparison_table.autofit = False
                
                # Set column widths
                widths = [Inches(1.5), Inches(1.5), Inches(1.5), Inches(1.5), Inches(1.5)]
                for i, width in enumerate(widths):
                    comparison_table.columns[i].width = width
                
                # Headers
                headers = ['Category', 'Missing Group\n(n, %)', 'Complete Group\n(n, %)', 'Missing Rate\n(n/total, %)', 'Total (n)']
                for i, header in enumerate(headers):
                    cell = comparison_table.rows[0].cells[i]
                    cell.text = header
                    cell.paragraphs[0].runs[0].font.bold = True
                    cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
                
                # Data
                for i, category in enumerate(all_categories):
                    row = comparison_table.rows[i + 1].cells
                    row[0].text = str(category)
                    
                    # Calculate counts and percentages
                    missing_count = (missing_valid == category).sum()
                    complete_count = (complete_valid == category).sum()
                    total_count = missing_count + complete_count
                    
                    missing_pct = (missing_count / len(missing_valid)) * 100 if len(missing_valid) > 0 else 0
                    complete_pct = (complete_count / len(complete_valid)) * 100 if len(complete_valid) > 0 else 0
                    missing_rate = (missing_count / total_count) * 100 if total_count > 0 else 0
                    
                    row[1].text = f"{missing_count:,}\n({missing_pct:.1f}%)"
                    row[2].text = f"{complete_count:,}\n({complete_pct:.1f}%)"
                    row[3].text = f"{missing_count:,}/{total_count:,}\n({missing_rate:.1f}%)"
                    row[4].text = f"{total_count:,}"
                    
                    # Center align all cells
                    for col_idx in range(5):
                        row[col_idx].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
                
                doc.add_paragraph()
                
                # Add interpretation
                doc.add_paragraph(f"Interpretation: {race_var['Interpretation']}")
                doc.add_paragraph()
    
    # 3.4 Summary of Key Findings
    doc.add_heading('3.4 Summary of Key Findings', level=2)
    
    findings_text = f"""
The analysis identified {len(significant_vars_list)} variables with statistically significant differences (p < 0.05):
• {len(numeric_vars_sig)} numeric variables show significant mean differences
• {len(categorical_vars_sig)} categorical variables show significant distribution differences
• {len(race_vars_sig)} race-related variables show significant differences

Top 10 Most Significant Variables:
"""
    
    # Add top 10 most significant findings
    sorted_vars = sorted(significant_vars_list, key=lambda x: x['P_Value'])[:10]
    for i, var in enumerate(sorted_vars, 1):
        if var['Type'] == 'Numeric':
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
        else:
            findings_text += f"\n{i}. {var['Variable']}: {var['Statistic']} vs {var['Comparison']} (p={var['P_Value']:.4f})"
    
    doc.add_paragraph(findings_text.strip())

# Add visualization
doc.add_heading('4. Statistical Visualizations', level=1)
if os.path.exists(output_png):
    doc.add_picture(output_png, width=Inches(6.5))
    doc.add_paragraph(f'Figure 1. IUPAC compliant statistical visualization of missing {TARGET_VARIABLE} patterns.')

# Recommendations
doc.add_heading('5. Recommendations', level=1)

if sig_count > 5:
    doc.add_paragraph('Based on the statistical analysis, the missing data exhibits SYSTEMATIC patterns:', style='List Bullet')
    doc.add_paragraph('Consider treating missing as a separate category in analysis', style='List Bullet')
    doc.add_paragraph('Use multiple imputation methods (MICE, EM algorithm)', style='List Bullet')
    doc.add_paragraph('Investigate the following key variables for common themes:', style='List Bullet')
    
    # Add list of significant variables with counts
    for var in significant_vars_list[:10]:
        if var['Type'] == 'Numeric':
            doc.add_paragraph(f'  - {var["Variable"]}: {var["Statistic"]} vs {var["Comparison"]} (p={var["P_Value"]:.4f})', style='List Bullet')
        else:
            doc.add_paragraph(f'  - {var["Variable"]}: {var["Statistic"]} vs {var["Comparison"]} (p={var["P_Value"]:.4f})', style='List Bullet')
    
    doc.add_paragraph('Document missingness patterns in methodology section', style='List Bullet')
    
elif sig_count > 2:
    doc.add_paragraph('Based on the statistical analysis, the missing data shows MODERATE systematic patterns:', style='List Bullet')
    doc.add_paragraph('Consider multiple imputation or sensitivity analysis', style='List Bullet')
    doc.add_paragraph('Document the patterns in your methodology', style='List Bullet')
    doc.add_paragraph('Validate findings with domain experts', style='List Bullet')
    
else:
    doc.add_paragraph('Based on the statistical analysis, the missing data appears RELATIVELY RANDOM:', style='List Bullet')
    doc.add_paragraph('Simple imputation methods may be acceptable (mean, median, mode)', style='List Bullet')
    doc.add_paragraph('Listwise deletion may be appropriate if missing rate is low', style='List Bullet')
    doc.add_paragraph('Still verify randomness with domain knowledge', style='List Bullet')

# Methodology
doc.add_heading('6. Methodology (IUPAC Compliant)', level=1)
doc.add_paragraph('Analysis performed according to IUPAC guidelines for data presentation:')
doc.add_paragraph('• Missing values reported as both count (n) and percentage (%)', style='List Bullet')
doc.add_paragraph('• Tables formatted with thin borders and alternating row shading', style='List Bullet')
doc.add_paragraph('• Sans-serif fonts (Arial) used for optimal readability', style='List Bullet')
doc.add_paragraph('• Statistical tests: t-test (numeric), χ² test (categorical)', style='List Bullet')
doc.add_paragraph('• Significance threshold: α = 0.05', style='List Bullet')
doc.add_paragraph('• Effect sizes reported where applicable', style='List Bullet')
doc.add_paragraph('• Figures saved at 300 DPI for publication quality', style='List Bullet')
doc.add_paragraph('• Sample sizes (n) reported for all comparisons', style='List Bullet')

# Add footer with page numbers
section = doc.sections[0]
footer = section.footer
footer_para = footer.paragraphs[0]
footer_para.text = f"{TARGET_VARIABLE} Missing Analysis | Generated: {datetime.now().strftime('%Y-%m-%d')} | Page "
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add page number field
run = footer_para.add_run()
fldChar1 = OxmlElement('w:fldChar')
fldChar1.set(qn('w:fldCharType'), 'begin')
run._r.append(fldChar1)

instrText = OxmlElement('w:instrText')
instrText.text = "PAGE"
run._r.append(instrText)

fldChar2 = OxmlElement('w:fldChar')
fldChar2.set(qn('w:fldCharType'), 'end')
run._r.append(fldChar2)

# Save document
output_docx = f'missing_{safe_filename_base}_iupac_report_{timestamp}.docx'
try:
    doc.save(output_docx)
    print(f"✅ Created IUPAC compliant Word document: {output_docx}")
except PermissionError as e:
    alt_filename = f'missing_{safe_filename_base}_iupac_report_{timestamp}_new.docx'
    doc.save(alt_filename)
    print(f"✅ Saved with alternative filename: {alt_filename}")
    output_docx = alt_filename

# ============================================
# 5. FINAL SUMMARY
# ============================================

print(f"\n{'='*80}")
print(f"📋 IUPAC COMPLIANT ANALYSIS COMPLETE")
print(f"{'='*80}")

print(f"\n📊 FILES GENERATED (IUPAC Compliant):")
print(f"   • Excel Report: {output_file}")
print(f"   • Visualization: {output_png}")
print(f"   • Word Report: {output_docx}")

print(f"\n📈 STATISTICAL SUMMARY:")
print(f"   • Missing {TARGET_VARIABLE}: {len(missing_data):,} ({missing_pct:.1f}%)")
print(f"   • Variables with significant differences: {sig_count}")
print(f"     - Numeric: {numeric_sig_count}")
print(f"     - Categorical: {categorical_sig_count}")
print(f"     - Race variables: {len([v for v in significant_vars_list if 'Race' in v['Variable']])}")

if sig_count > 5:
    pattern = "SYSTEMATIC PATTERN"
elif sig_count > 2:
    pattern = "MODERATE PATTERN"
else:
    pattern = "RANDOM PATTERN"

print(f"   • Missingness pattern: {pattern}")

print(f"\n📋 TOP SIGNIFICANT VARIABLES (with counts):")
for var in significant_vars_list[:10]:
    if var['Type'] == 'Numeric':
        print(f"   • {var['Variable']}: {var['Statistic']} vs {var['Comparison']}, p={var['P_Value']:.4f}")
    else:
        print(f"   • {var['Variable']}: {var['Statistic']} vs {var['Comparison']}, p={var['P_Value']:.4f}")

print(f"\n{'='*80}")
print(f"✅ All IUPAC compliant files generated successfully")
print(f"{'='*80}")


📊 COMPREHENSIVE STATISTICAL ANALYSIS OF MISSING Multiple_Birth_Status PATTERNS
(IUPAC Compliant Output)

📌 DATASET OVERVIEW:
   • Total records with missing Multiple_Birth_Status: 363,824
   • Total records with complete Multiple_Birth_Status: 57
   • Missing percentage: 99.98%

📌 PART 1: UNIVARIATE ANALYSIS - DETAILED STATISTICS FOR EACH VARIABLE

Analyzing 17 variables...

✅ Analysis complete. Found 8 variables with significant differences (p < 0.05)

📋 SIGNIFICANT VARIABLES (p < 0.05):
   • Registered_Month: Most common='October' (32,558/363,824, 8.9%) vs Complete: Most common='June' (12/57, 21.1%), p=0.0000
   • Registered_District_2.0: Mean=41.60 (n=363,824) vs Complete Mean=33.26 (n=57), p=0.0329
   • Registered_District: Most common='Colombo' (55,453/363,824, 15.2%) vs Complete: Most common='Colombo' (21/57, 36.8%), p=0.0000
   • Birh_Year: Mean=2009.57 (n=363,824) vs Complete Mean=2009.95 (n=57), p=0.0000
   • Birth_Month: Most common='October' (33,694/363,824, 9.3%) vs Comple